# Know Your Data: Exploring and Preparing Image Datasets at the Wildwood Farm Park

**Objective:** run the setup, meet the park, and see the three things you will be able to do at the end.

<table width="100%"><tr><td bgcolor="#0B6E79"><font color="#FFFFFF">
<font size="5"><b>Know Your Data: Exploring and Preparing Image Datasets
at the Wildwood Farm Park</b></font><br><br>
<b>At the end of this notebook you will be able to:</b>
<ol>
<li>Inspect an image dataset qualitatively and quantitatively, and say whether it fits a task.</li>
<li>Find and fix common dataset errors (duplicates, flipped/rotated copies, broken files, bad labels or
masks) before they leak into training.</li>
<li>Choose a split and a metric that match the use case, and explain how imbalance, object size and leakage
look different for classification, detection and segmentation.</li>
</ol>
Questions are marked 🟢 beginner · 🟡 intermediate · 🔴 advanced (optional).
</font></td></tr></table>

**Before you start:** you need a Google account (this notebook runs in Colab, a CPU runtime is enough),
Python basics, and the lecture on image datasets. No model is trained here; that is next week.
The notebook follows the first part of Karpathy's
[*A Recipe for Training Neural Networks*](https://karpathy.github.io/2019/04/25/recipe/):
know your data before you touch a model.

## 0. Setup & the park

**The Wildwood Farm Park** has cameras in four places, and it wants three things from them.

<table width="100%"><tr>
<td align="center" width="25%"><font size="6">🐑🐴</font><br><b>Farm paddock</b><br>sheep, horses</td>
<td align="center" width="25%"><font size="6">🐦</font><br><b>Aviary pond</b><br>birds</td>
<td align="center" width="25%"><font size="6">🦓</font><br><b>Savanna enclosure</b><br>zebras</td>
<td align="center" width="25%"><font size="6">🐱🐶</font><br><b>Visitor yard</b><br>cats and dogs wander
through</td>
</tr></table>

1. **Recognise** (classification): "who is in front of this camera?" The case that matters is **a dog in the
   sheep paddock**, which triggers an alert. Missing that dog is worse than a false alarm.
2. **Count** (detection): a daily headcount of sheep, zebras and birds. Far-away birds are tiny.
3. **Outline** (segmentation): how much of each enclosure each animal uses. Birds cover few pixels.

The park has no training data yet, so we practise on a **Play Dataset**: COCO 2017 photos of the same six
animals (cat, dog, bird, horse, sheep, zebra), each photo showing one animal class. Everything you learn
here is a checklist you will reuse on your own project dataset in the last section.

**Run the next cell once.** It installs the one package Colab lacks, fixes the random seed, sets the
notebook's colours and defines `use()`, the helper that lets you write small functions without ever
blocking "Run all".

In [ ]:
# @title Setup (run me first) { display-mode: "form" }
import ast
import inspect
import os
import random
import sys
import textwrap

import matplotlib as mpl
import matplotlib.pyplot as plt  # noqa: F401  (every later section plots with plt)
import numpy as np

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("NB_SMOKE") == "1"  # automated tests: tiny data, CPU, seconds. Never set this by hand.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if IN_COLAB:  # Jupytext turns the commented magic below into a real `%pip install -q imagehash` (N-021)
    pass
    %pip install -q imagehash

# --- Theme (SPEC L-003): white background, dark teal text, light grid, one colour per class everywhere ---
CLASSES = ["cat", "dog", "bird", "horse", "sheep", "zebra"]
CLASS_COLORS = {
    "cat": "#E07A5F",
    "dog": "#3D405B",
    "bird": "#F2CC8F",
    "horse": "#81B29A",
    "sheep": "#9C89B8",
    "zebra": "#2A9D8F",
}
INK = "#21434A"
mpl.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "text.color": INK,
        "axes.labelcolor": INK,
        "axes.edgecolor": INK,
        "axes.titlecolor": INK,
        "xtick.color": INK,
        "ytick.color": INK,
        "axes.grid": True,
        "grid.color": "#DDE5E7",
        "grid.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.prop_cycle": mpl.cycler(color=[CLASS_COLORS[c] for c in CLASSES]),
        "figure.dpi": 100,
        "font.size": 11,
    }
)

# --- Results every section writes into (SPEC §9) ---
RESULTS = {}
RESULTS["class_colors"] = dict(CLASS_COLORS)


# --- use(): run a learner's function next to the reference solution (N-029) ---
def _is_stub(fn) -> bool:
    """True if fn's body is only a docstring plus `...` or `pass` (the learner has not written it yet)."""
    try:
        node = ast.parse(textwrap.dedent(inspect.getsource(fn))).body[0]
        body = node.body
    except (OSError, TypeError, SyntaxError, IndexError, AttributeError):
        return False
    if body and isinstance(body[0], ast.Expr) and isinstance(getattr(body[0].value, "value", None), str):
        body = body[1:]  # skip the docstring
    return all(
        isinstance(n, ast.Pass) or (isinstance(n, ast.Expr) and getattr(n.value, "value", None) is Ellipsis)
        for n in body
    )


def _same(a, b) -> bool:
    if hasattr(a, "equals"):  # pandas
        return bool(a.equals(b))
    try:
        if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
            return bool(np.array_equal(a, b))
        return bool(a == b)
    except Exception:
        return False


def _short(x, n=60) -> str:
    s = repr(x)
    return s if len(s) <= n else s[: n - 1] + "…"


def _diff(mine, ref) -> str:
    """One short line on how `mine` differs from `ref` (e.g. 'cat: 12 vs 14')."""
    if isinstance(mine, dict) and isinstance(ref, dict):
        keys = list(ref) + [k for k in mine if k not in ref]
        bad = [k for k in keys if not _same(mine.get(k, "—"), ref.get(k, "—"))]
        return ", ".join(f"{k}: {_short(mine.get(k, '—'))} vs {_short(ref.get(k, '—'))}" for k in bad[:4])
    if isinstance(mine, (list, tuple, set)) and isinstance(ref, (list, tuple, set)):
        try:
            extra, missing = set(mine) - set(ref), set(ref) - set(mine)
            parts = []
            if missing:
                parts.append(f"missing {_short(sorted(missing, key=str))}")
            if extra:
                parts.append(f"extra {_short(sorted(extra, key=str))}")
            return " · ".join(parts) or f"same items, different order or length ({len(mine)} vs {len(ref)})"
        except TypeError:
            pass
    return f"{_short(mine)} vs {_short(ref)}"


def use(fn, *args, reference, **kwargs):
    """Run `fn(*args, **kwargs)` next to `reference(*args, **kwargs)`; ALWAYS return the reference's result.

    - stub (body is `...`) or NotImplementedError → "using the reference solution";
    - otherwise ✅ when both agree, ❌ with a short diff when they do not.
    The notebook continues with the reference either way, so "Run all" never breaks on your code.
    """
    expected = reference(*args, **kwargs)
    name = getattr(fn, "__name__", "your function")
    if _is_stub(fn):
        print(f"✍️  {name}: not written yet, using the reference solution.")
        return expected
    try:
        mine = fn(*args, **kwargs)
    except NotImplementedError:
        print(f"✍️  {name}: not written yet, using the reference solution.")
        return expected
    except Exception as err:  # a learner's bug must not stop the notebook
        print(f"❌ {name} raised {type(err).__name__}: {err} · using the reference solution.")
        return expected
    if _same(mine, expected):
        print(f"✅ {name} matches the reference solution.")
    else:
        print(f"❌ {name} differs from the reference: {_diff(mine, expected)} · using the reference.")
    return expected


print(f"Colab: {IN_COLAB} · smoke mode: {SMOKE} · seed: {SEED} · imagehash: ", end="")
try:
    import imagehash  # noqa: F401

    print("ok")
except ImportError:
    print("missing (run this cell again)")

### How the ✍️ cells work

About four times in this notebook you write a small function. Each one comes with its signature and a
`use(my_function, …, reference=…)` line that compares your result with the reference solution and
then continues with the reference, so an unfinished or wrong function never stops the notebook.
Run the cell below as is, then fill in the body and run it again to see the message change.

In [ ]:
def double(x):
    """Return twice x."""
    ...


def _ref_double(x):
    return 2 * x


result = use(double, 21, reference=_ref_double)
print("the notebook continues with:", result)

### The Play Dataset

The next cell fetches the annotation subset we prepared (`data/coco_animals_subset.json`, COCO 2017 val,
six animal classes, each image shows exactly one of them) and downloads the photos themselves from
`images.cocodataset.org`. **The first run takes a minute or two** (about 115 MB); afterwards the images
are cached and the cell is instant. The result is two tidy tables, `images` (one row per photo) and
`annotations` (one row per animal: its box and mask).

In [ ]:
# @title Data helpers (download with cache, build the tables) { display-mode: "form" }
import json
import urllib.error
import urllib.request
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd

# Where the committed manifests live. In Colab they are fetched from the repository; locally they are read
# from ./data. ⬇️ DATA_BASE_URL: change this if the repository moves or you work on a fork.
DATA_BASE_URL = "https://raw.githubusercontent.com/FHNW-VISED/eda-teaching-notebook/main/data/"
DATA_DIR = Path("data")
IMAGE_CACHE = DATA_DIR / "cache" / "coco"
SMOKE_IMAGES_PER_CLASS = 6


def _github_headers() -> dict:
    """Authorization header from the Colab secret GITHUB_TOKEN, if the repository is private (SPEC §5)."""
    if not IN_COLAB:
        return {}
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
        return {"Authorization": f"token {token}"} if token else {}
    except Exception:
        return {}


def download_file(url: str, dest: Path, headers: dict | None = None, attempts: int = 3) -> Path | None:
    """Download `url` to `dest` once; a half-finished download is written to `.part` first (N-019)."""
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(attempts):
        try:
            req = urllib.request.Request(url, headers=headers or {})
            with urllib.request.urlopen(req, timeout=60) as resp, open(part, "wb") as fh:
                fh.write(resp.read())
            part.rename(dest)
            return dest
        except (urllib.error.URLError, TimeoutError, OSError):
            if attempt == attempts - 1:
                part.unlink(missing_ok=True)
                return None
    return None


def fetch_manifest(name: str) -> Path:
    """Return the local path of a committed manifest, downloading it from DATA_BASE_URL if needed."""
    path = DATA_DIR / name
    if not path.exists():
        if download_file(DATA_BASE_URL + name, path, headers=_github_headers()) is None:
            raise RuntimeError(
                f"Could not download {name} from {DATA_BASE_URL}. Check DATA_BASE_URL, or add a GITHUB_TOKEN "
                "secret in Colab if the repository is private."
            )
    return path


def download_images(rows: list[dict], workers: int = 16) -> dict[int, Path]:
    """Download every image in `rows` ('id', 'coco_url', 'file_name') into IMAGE_CACHE; one progress line."""
    done, paths = 0, {}

    def one(row):
        return row["id"], download_file(row["coco_url"], IMAGE_CACHE / row["file_name"])

    with ThreadPoolExecutor(max_workers=workers) as pool:
        for image_id, path in pool.map(one, rows):
            done += 1
            if path is not None:
                paths[image_id] = path
            print(f"\r  images ready: {done}/{len(rows)}", end="", flush=True)
    print()
    return paths


def build_tables(coco: dict, paths: dict[int, Path]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Tidy tables from a COCO-format dict: `images` (one row per photo), `annotations` (one per animal)."""
    cat_name = {c["id"]: c["name"] for c in coco["categories"]}
    lic_name = {lic["id"]: lic["name"] for lic in coco.get("licenses", [])}
    anns = pd.DataFrame(
        {
            "id": a["id"],
            "image_id": a["image_id"],
            "class": cat_name[a["category_id"]],
            "bbox": a["bbox"],
            "area": a["area"],
            "segmentation": a["segmentation"],
            "iscrowd": a["iscrowd"],
        }
        for a in coco["annotations"]
        if a["image_id"] in paths
    )
    per_image_class = anns.groupby("image_id")["class"].first()
    n_animals = anns.groupby("image_id").size()
    imgs = pd.DataFrame(
        {
            "id": im["id"],
            "file": im["file_name"],
            "path": str(paths[im["id"]]),
            "width": im["width"],
            "height": im["height"],
            "class": per_image_class.get(im["id"]),
            "n_animals": int(n_animals.get(im["id"], 0)),
            "license": lic_name.get(im["license"], str(im["license"])),
        }
        for im in coco["images"]
        if im["id"] in paths
    )
    imgs = imgs.sort_values("id").reset_index(drop=True)
    anns = anns.sort_values("id").reset_index(drop=True)
    return imgs, anns

In [ ]:
# Load the manifest, pick the images (all of them, or 6 per class in smoke mode), download, build the tables.
COCO_RAW = json.loads(fetch_manifest("coco_animals_subset.json").read_text())
_cat_name = {c["id"]: c["name"] for c in COCO_RAW["categories"]}
_image_class = {a["image_id"]: _cat_name[a["category_id"]] for a in COCO_RAW["annotations"]}

_rows = COCO_RAW["images"]
if SMOKE:  # same code path, fewer images (N-014)
    _rng = random.Random(SEED)
    _rows = []
    for c in CLASSES:
        _of_class = [im for im in COCO_RAW["images"] if _image_class[im["id"]] == c]
        _rows += sorted(_rng.sample(_of_class, SMOKE_IMAGES_PER_CLASS), key=lambda im: im["id"])

print(f"{len(_rows)} images to fetch from images.cocodataset.org (cached in {IMAGE_CACHE}/)")
_paths = download_images(_rows)
if len(_paths) < len(_rows):
    print(f"  ⚠️ {len(_rows) - len(_paths)} images could not be downloaded and are left out; run again.")

images, annotations = build_tables(COCO_RAW, _paths)
RESULTS["n_images"] = int(len(images))

_counts = Counter(images["class"])
print(f"images: {len(images)} · animals (annotations): {len(annotations)}")
print("per class: " + " · ".join(f"{c} {_counts[c]}" for c in CLASSES))
images.head()

Each row of `images` is one photo with its size, its class and how many animals it holds; `annotations`
has one row per animal with its box and mask. Look at the `n_animals` column: several photos hold more
than one animal, which already hints at a question we will meet in section 4.

## 1. Image datasets

**Objective:** know where image datasets come from and what makes a good one, then hold the Play Dataset
against that checklist.

The park has no training data yet. Before anyone mounts cameras and hires annotators, the first question is
always: *does a usable dataset already exist?* These are the places to look, and the tools to label your own
images when nothing fits.

**Find a dataset:**
[Kaggle datasets](https://www.kaggle.com/datasets) ·
[Google Dataset Search](https://datasetsearch.research.google.com) ·
[V7 open datasets](https://www.v7labs.com/open-datasets) ·
[Roboflow Universe](https://universe.roboflow.com) ·
[LILA](https://lila.science) (wildlife and camera-trap data) ·
[V7 blog on computer-vision datasets](https://www.v7labs.com/blog) ·
[torchvision datasets](https://pytorch.org/vision/stable/datasets.html) (the classics, one import away).

**Label your own images:**
[Supervisely](https://supervisely.com) ·
[V7](https://www.v7labs.com) (free for education) ·
[Roboflow Annotate](https://roboflow.com/annotate) ·
[Label Studio](https://labelstud.io) (open source).

Our Play Dataset is a slice of [COCO](https://cocodataset.org), and the woodland trail-camera images of
section 6 come from LILA. Part of "finding" a dataset is reading its licence: COCO's annotations are
CC BY 4.0, but every photo keeps its own Flickr licence, which is why this notebook downloads the images at
runtime instead of shipping them.

Having a dataset is not the same as having a *good* one. This is the checklist from the lecture:

<table width="100%"><tr><td bgcolor="#E3F2FD"><font color="#0D47A1">
<b>📘 Theory · What makes a good dataset</b><br>
<ul>
<li><b>Does it fit the question?</b> The images, the classes and the setting must match the job you want the
model to do.</li>
<li><b>Enough samples:</b> a few hundred to a few thousand images, depending on how many classes you
have.</li>
<li><b>2–10 classes.</b></li>
<li><b>About 500×500 px.</b> Check the input size of the models you plan to use.</li>
<li><b>Classes, distributions, imbalance, bias.</b> Know them, and take them into account in training
(for example with a weighted loss).</li>
<li><b>Variability matches the target population.</b> Lighting, viewpoints, backgrounds and seasons in the
data should look like the ones the model will meet.</li>
<li><b>Labels are present and validated.</b></li>
<li><b>Objects are identifiable and separable by eye.</b> If you cannot tell them apart, the model will
struggle too.</li>
<li><b>What do experts and the literature say</b> about this data and this task?</li>
<li><b>Get a qualitative and a quantitative sense of the data</b> (Karpathy, 2019): look at many images,
then measure.</li>
</ul>
</font></td></tr></table>

The next cell goes through the checklist for the Play Dataset. Where the tables from section 0 can answer
a bullet it computes the answer (✅ fine, ⚠️ needs attention); where only your eyes can, it says so (👀)
and points to the section that will look.

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
Predict before you run: the park wants to recognise, count and outline six animals from fixed cameras.
</font></td></tr></table>

**Q1.** 🟢 Which bullets do you expect the Play Dataset to pass (✅) as it is, and which one worries you
most for the dog-in-the-paddock alert? Write one line, then run the cell.

In [ ]:
# Q1  Apply the checklist to the Play Dataset. Every number comes from the `images` table of section 0.
n_images = len(images)
counts = images["class"].value_counts().reindex(CLASSES).fillna(0).astype(int)
n_classes = int((counts > 0).sum())
imbalance = counts.max() / max(counts.min(), 1)
median_w, median_h = int(images["width"].median()), int(images["height"].median())
share_labelled = float((images["n_animals"] > 0).mean())
share_several = float((images["n_animals"] > 1).mean())

checklist = [
    (
        "⚠️",
        "Fits the question?",
        "the park's six animals, but Flickr holiday photos, not fixed cameras → judge in section 2",
    ),
    (
        "✅" if n_images >= 300 else "⚠️",
        "A few hundred to a few thousand samples",
        f"{n_images} images" + ("" if n_images >= 300 else " (a tiny test sample; the full set has ~600)"),
    ),
    ("✅" if 2 <= n_classes <= 10 else "⚠️", "2–10 classes", f"{n_classes}: " + ", ".join(CLASSES)),
    (
        "✅" if min(median_w, median_h) >= 224 else "⚠️",
        "About 500×500 px",
        f"median {median_w}×{median_h} px; most classifiers take 224–512 px, so there is room to resize",
    ),
    (
        "⚠️" if imbalance > 2 else "✅",
        "Classes, distributions, imbalance, bias",
        " · ".join(f"{c} {counts[c]}" for c in CLASSES)
        + f" → largest / smallest = {imbalance:.1f}× (weighted loss? section 4)",
    ),
    ("👀", "Variability matches the target population", "lighting, distance, backgrounds: look in section 2"),
    (
        "✅" if share_labelled == 1 else "⚠️",
        "Labels present",
        f"{share_labelled:.0%} of images have at least one box and mask",
    ),
    (
        "⚠️",
        "Labels validated",
        "COCO-ReM (Singh et al., 2024) found ~2.4 % of val masks poor, plus duplicate masks → sections 5, 9",
    ),
    (
        "👀",
        "Objects identifiable and separable by eye",
        f"{share_several:.0%} of images hold several animals; check the grids in section 2",
    ),
    (
        "📚",
        "What experts and the literature say",
        "Lin et al. 2014 (COCO) and Singh et al. 2024 (COCO-ReM) document the data and its mask errors",
    ),
    ("📓", "Qualitative and quantitative sense (Karpathy 2019)", "sections 2–4 of this notebook"),
]
for status, bullet, finding in checklist:
    print(f"{status} {bullet}: {finding}")

Read the list top to bottom: the size, class count and resolution pass on their own, and everything that
needs attention points to a later section of this notebook, which is the order we will follow.

**Worked answer (Q1).** Sample size, number of classes and resolution pass without any work. The bullets
that matter for the alert are *fit* and *imbalance*: holiday photos of dogs are not a camera pointed at a
paddock, and on the full set the biggest class has more than twice the images of the smallest, so a model
can score well while ignoring a rare animal. Section 4 makes that concrete for the dog and section 7 picks
the metric for it.

## 2. Become one with the data

**Objective:** look at many images before computing anything, and write down what you see.

Section 1 ended with three 👀 bullets that no table can answer: does the data *fit* the park's cameras, does
its *variability* match what the cameras will see, and are the animals *identifiable by eye*? The only tool
for those is your eyes, and this is the step most people skip. Karpathy puts it first in his recipe:

<table width="100%"><tr><td bgcolor="#E8F5E9"><font color="#1B5E20">
<b>🧾 Recipe · Step 1: Become one with the data</b><br>
<i>"The first step to training a neural net is to not touch any neural net code at all and instead begin by
thoroughly inspecting your data. This step is critical. I like to spend copious amount of time (measured in
units of hours) scanning through thousands of examples, understanding their distribution and looking for
patterns."</i><br>
Andrej Karpathy, <a href="https://karpathy.github.io/2019/04/25/recipe/">A Recipe for Training Neural
Networks</a> (2019).
</font></td></tr></table>

We will not spend hours, but we will spend a few minutes. The first cell draws one row of photos per class,
each framed in its class colour (the same colour every plot in this notebook uses for that class). Scroll
slowly: look at the backgrounds, the distance to the animal, the lighting, and whether anything else is in
the photo.

In [ ]:
# @title Grid helpers { display-mode: "form" }
from IPython.display import display
from PIL import Image


def load_thumb(path, size: int = 256) -> Image.Image:
    """Open an image as a small RGB thumbnail (fast: JPEG decodes at reduced size)."""
    im = Image.open(path)
    im.draft("RGB", (size, size))
    im = im.convert("RGB")
    im.thumbnail((size, size))
    return im


def sample_rows(class_name: str, n: int) -> pd.DataFrame:
    """A fixed random sample of `n` rows of `images` for one class (seeded: re-runs show the same photos)."""
    of_class = images[images["class"] == class_name]
    n = min(n, len(of_class))
    idx = random.Random(SEED).sample(sorted(of_class.index), n)
    return of_class.loc[sorted(idx)]


def frame(ax, color: str, width: float = 4) -> None:
    """Draw a coloured frame around an image axis and hide ticks and grid."""
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(color)
        spine.set_linewidth(width)


def show_grid(class_name: str, n: int, cols: int = 6, label_animals: bool = True) -> None:
    """Show `n` photos of one class in a grid, framed in CLASS_COLORS[class_name]."""
    rows_df = sample_rows(class_name, n)
    n = len(rows_df)
    n_rows = max(1, -(-n // cols))
    fig, axes = plt.subplots(n_rows, cols, figsize=(2.2 * cols, 2.4 * n_rows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, (_, row) in zip(axes, rows_df.iterrows(), strict=False):
        ax.axis("on")
        ax.imshow(load_thumb(row["path"]))
        frame(ax, CLASS_COLORS[class_name])
        if label_animals:
            plural = "s" if row["n_animals"] != 1 else ""
            ax.set_title(f"{row['n_animals']} {class_name}{plural}", fontsize=8)
    fig.suptitle(f"{class_name} · {n} of {int((images['class'] == class_name).sum())} photos", color=INK)
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def show_all_classes(n_per_class: int) -> None:
    """One row per class, `n_per_class` photos each, every row framed in its class colour."""
    fig, axes = plt.subplots(len(CLASSES), n_per_class, figsize=(2.0 * n_per_class, 2.1 * len(CLASSES)))
    axes = np.atleast_2d(axes)
    for r, class_name in enumerate(CLASSES):
        rows_df = sample_rows(class_name, n_per_class)
        for c in range(n_per_class):
            ax = axes[r, c]
            ax.axis("off")
            if c < len(rows_df):
                ax.axis("on")
                ax.imshow(load_thumb(rows_df.iloc[c]["path"]))
                frame(ax, CLASS_COLORS[class_name])
            if c == 0:
                ax.set_ylabel(class_name, color=CLASS_COLORS[class_name], fontsize=12, fontweight="bold")
    fig.tight_layout()
    display(fig)
    plt.close(fig)

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
Predict before you run: the park's cameras are fixed, outdoors, and often far from the animal.
</font></td></tr></table>

**Q2.** 🟢 Which class do you expect to look *least* like a fixed park camera's view, and why? One line,
then run the cell.

In [ ]:
# Q2  One row per class: a fixed sample of photos, framed in the class colour used throughout the notebook.
N_PER_CLASS = 3 if SMOKE else 6  # fewer thumbnails in smoke mode, same code (N-014)
show_all_classes(N_PER_CLASS)

💡 Even six photos per class show that these are holiday snapshots, not camera-trap frames: close-ups,
people, sofas and streets, with the animal usually centred and filling the frame.

**Worked answer (Q2).** Cats and dogs are the farthest from a park camera: they are photographed indoors,
up close, often with people and furniture. Sheep, horses and zebras are at least outdoors and sometimes far
away, which is closer to what the paddock and savanna cameras will see.

Now pick one class and look at more of it. Twelve photos is enough to start noticing patterns; raise it if
you have time.

<table width="100%"><tr><td bgcolor="#FFF4D6"><font color="#7A4A00">
<b>✏️ Your turn · Pick a class</b><br>
Set <code>CLASS_TO_SHOW</code> to one of <code>cat, dog, bird, horse, sheep, zebra</code> and
<code>N_SHOW</code> to how many photos you want (up to the class size), then run the cell. Try at least
<code>"dog"</code> (the alert class) and <code>"bird"</code> (the tiny-box class).
</font></td></tr></table>

In [ ]:
# ✏️ YOUR TURN
CLASS_TO_SHOW = "dog"
N_SHOW = 12

if CLASS_TO_SHOW not in CLASSES:
    raise SystemExit(f"CLASS_TO_SHOW must be one of {CLASSES}, not {CLASS_TO_SHOW!r}")
if not isinstance(N_SHOW, int) or N_SHOW < 1:
    raise SystemExit(f"N_SHOW must be a positive integer, not {N_SHOW!r}")
n_show = min(N_SHOW, 6) if SMOKE else N_SHOW  # smoke mode caps the grid, same code path (N-014)
show_grid(CLASS_TO_SHOW, n_show)

💡 The title of each photo says how many animals of that class it holds: more than one is common, so "one
label per image" is already a simplification, and section 4 asks which label a photo with several animals
should get.

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
Karpathy's step ends with writing things down. Do the same: edit the next markdown cell (double-click it).
</font></td></tr></table>

**Q3.** 🟢 Write three observations about the photos you just looked at. Good observations are specific
("many dog photos are indoors with a person's legs in the frame"), and each one hints at something the park
should worry about or check later (fit to the cameras, lighting, distance, several animals, people, odd
photos).

**My three observations** (double-click to edit):

1. …
2. …
3. …

**Worked answer (Q3)**, for the dog grid on the full set. Yours will differ; that is the point.

1. **Setting:** most dog photos are indoors or in a street, taken by the owner from a metre away. A paddock
   camera sees a small dog far away in grass. The model will have to generalise from close-ups to that
   (section 11's question "does it fit?").
2. **People:** a person, a hand or a leash appears in many photos. A model can learn "person + leash = dog"
   and miss the stray dog alone in the paddock.
3. **Several animals and odd photos:** some photos hold two or three dogs, some show a dog in a costume,
   on a skateboard or on a screen. Each of these is a labelling decision (sections 4 and 5) before it is a
   modelling problem.

Keep your three lines: section 11 asks you for the same three on your own dataset, and the rest of this
notebook turns the observations above into numbers, one at a time.

## 3. Image inspection

**Objective:** measure sizes, aspect ratios, colour modes and colour statistics, and learn to read a cluster
of odd values as a clue about where the photos came from.

Your eyes found patterns in section 2. Now the computer looks at every photo once and writes down a handful
of numbers per image: its size, its shape, how its colours are stored and how bright it is. None of these
numbers needs a label, so this is the first check you can run on *any* folder of images, including your own
project dataset in section 11. Karpathy's recipe says why this pays off:

<table width="100%"><tr><td bgcolor="#E8F5E9"><font color="#1B5E20">
<b>🧾 Recipe · Step 1, continued: search, filter, sort</b><br>
<i>"Once you get a qualitative sense it is also a good idea to write some simple code to search/filter/sort
by whatever you can think of (e.g. type of label, size of annotations, number of annotations, etc.) and
visualize their distributions and the outliers along any axis. The outliers especially almost always uncover
some bugs in data quality or preprocessing."</i><br> Andrej Karpathy, <a
href="https://karpathy.github.io/2019/04/25/recipe/">A Recipe for Training Neural Networks</a> (2019).
</font></td></tr></table>

The next cell (collapsed) defines one function that measures a single photo. The cell after it runs that
function on every photo in the Play Dataset and collects the numbers in a table called `image_stats`.

In [ ]:
# @title Measurement helpers { display-mode: "form" }
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.patches import Rectangle
from PIL import ImageDraw

HIST_BINS = 32
CHANNEL_COLORS = {"R": "#C0392B", "G": "#27AE60", "B": "#2980B9"}
COLOUR_MODES = ["RGB", "L", "RGBA", "CMYK"]  # the modes we always report, even when the count is 0


def measure_image(path, thumb: int) -> tuple[dict, np.ndarray]:
    """Open one photo and return (stats, channel_hist).

    stats: width, height, mode (as stored in the file), aspect (width / height), long_side, megapixels,
    brightness (mean grey level 0–255 on a thumbnail), contrast (its standard deviation), colourfulness
    (mean distance of the RGB channels from grey: 0 for a grayscale picture), mean_r / mean_g / mean_b.
    channel_hist: a (3, HIST_BINS) array, the density histogram of the R, G and B channels.
    """
    with Image.open(path) as im:
        mode, (w, h) = im.mode, im.size  # read these before decoding: they describe the file itself
        im.draft("RGB", (thumb, thumb))  # fast reduced-size JPEG decode (no effect on other formats)
        rgb = im.convert("RGB")
        rgb.thumbnail((thumb, thumb))
        arr = np.asarray(rgb, dtype=np.float32)
    grey = arr.mean(axis=2)
    channels = arr.reshape(-1, 3)
    hist = np.stack(
        [np.histogram(channels[:, k], bins=HIST_BINS, range=(0, 256), density=True)[0] for k in range(3)]
    )
    stats = {
        "width": w,
        "height": h,
        "mode": mode,
        "aspect": w / h,
        "long_side": max(w, h),
        "megapixels": w * h / 1e6,
        "brightness": float(grey.mean()),
        "contrast": float(grey.std()),
        "colourfulness": float(np.abs(channels - grey.reshape(-1, 1)).mean()),
        "mean_r": float(channels[:, 0].mean()),
        "mean_g": float(channels[:, 1].mean()),
        "mean_b": float(channels[:, 2].mean()),
    }
    return stats, hist


def show_rows(rows_df: pd.DataFrame, title: str, caption=None, cols: int = 6) -> None:
    """A small gallery of the photos in `rows_df` (rows of image_stats), each framed in its class colour.

    `caption(row) -> str` writes the title above each photo; by default the size and the colour mode.
    """
    if caption is None:

        def caption(row):
            return f"{row['class']} · {row['width']}×{row['height']} {row['mode']}"

    n = len(rows_df)
    if n == 0:
        print(f"{title}: no photos to show.")
        return
    cols = min(cols, n)
    n_rows = -(-n // cols)
    fig, axes = plt.subplots(n_rows, cols, figsize=(2.4 * cols, 2.7 * n_rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (_, row) in zip(axes.ravel(), rows_df.iterrows(), strict=False):
        ax.axis("on")
        ax.imshow(load_thumb(row["path"]))
        frame(ax, CLASS_COLORS[row["class"]])
        ax.set_title(caption(row), fontsize=8)
    fig.suptitle(title, color=INK)
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def show_extremes(stat: str, n: int = 4) -> None:
    """Gallery of the `n` photos with the lowest and the `n` with the highest value of `stat`.

    `stat` is a numeric column of `image_stats` (e.g. "brightness", "aspect", "megapixels", "colourfulness").
    """
    numeric = [c for c in image_stats.columns if pd.api.types.is_numeric_dtype(image_stats[c]) and c != "id"]
    if stat not in numeric:
        raise SystemExit(f"stat must be one of {numeric}, not {stat!r}")
    ordered = image_stats.sort_values(stat)
    n = max(1, min(n, len(ordered) // 2))

    def caption(row):
        return f"{row['class']} · {stat} = {row[stat]:.2f}\n{row['width']}×{row['height']} {row['mode']}"

    show_rows(ordered.head(n), f"lowest {stat}", caption, cols=n)
    show_rows(ordered.tail(n)[::-1], f"highest {stat}", caption, cols=n)

In [ ]:
# Measure every photo once (a few seconds: each JPEG is decoded at thumbnail size) and keep the numbers.
THUMB = 64 if SMOKE else 128  # smaller thumbnails in smoke mode, same code (N-014)

_records, _hists = [], []
for _, _row in images.iterrows():
    _stats, _hist = measure_image(_row["path"], THUMB)
    _records.append({"id": _row["id"], "class": _row["class"], "path": _row["path"], **_stats})
    _hists.append(_hist)
image_stats = pd.DataFrame(_records)
CHANNEL_HISTS = np.stack(_hists)  # (n_images, 3, HIST_BINS)
RESULTS["image_stats_n"] = int(len(image_stats))

_same_size = image_stats[["width", "height"]].to_numpy() == images[["width", "height"]].to_numpy()
_agree = int(_same_size.all(axis=1).sum())
print(f"measured {len(image_stats)} photos · the manifest's width/height matches the file for {_agree}")
STAT_COLUMNS = ["width", "height", "aspect", "megapixels", "brightness", "contrast", "colourfulness"]
image_stats[STAT_COLUMNS].describe().round(2)

💡 `describe()` is the quickest sanity check there is: look at `min` and `max` of every column before you
look at anything else. A `min` width of 0, a `max` of 6000 or a brightness of 254 each tell a story.

### Sizes and shapes

A model sees every photo at one fixed size, so the first question is how far the photos are from that.

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
Predict before you run. These photos were uploaded to Flickr by hundreds of people and collected by the COCO
team; the park's photos will come from a few fixed cameras.
</font></td></tr></table>

**Q4.** 🟢 Do you expect all photos to have the same size? Sketch (in your head) where the dots of a
width-vs-height scatter would land: one point, a line, a cloud? Then run the cell.

In [ ]:
# Q4  Left: every photo as a dot (width across, height up) in its class colour; the dashed line is a square
#     photo. Right: the aspect ratio width/height, stacked by class.
fig, (ax_sc, ax_hist) = plt.subplots(1, 2, figsize=(13, 4.8))
for c in CLASSES:
    s = image_stats[image_stats["class"] == c]
    ax_sc.scatter(s["width"], s["height"], s=22, alpha=0.65, color=CLASS_COLORS[c], label=c, edgecolor="none")
_lim = float(max(image_stats["width"].max(), image_stats["height"].max())) * 1.08
ax_sc.plot([0, _lim], [0, _lim], "--", color=INK, lw=1)
ax_sc.set_xlim(0, _lim)
ax_sc.set_ylim(0, _lim)
ax_sc.set_xlabel("width (px)")
ax_sc.set_ylabel("height (px)")
ax_sc.set_title("Image sizes")
ax_sc.legend(frameon=False, fontsize=9, markerscale=1.6)

_bins = np.linspace(image_stats["aspect"].min() * 0.98, image_stats["aspect"].max() * 1.02, 28)
ax_hist.hist(
    [image_stats.loc[image_stats["class"] == c, "aspect"] for c in CLASSES],
    bins=_bins,
    stacked=True,
    color=[CLASS_COLORS[c] for c in CLASSES],
    label=CLASSES,
)
for ratio, name in [(3 / 4, "3:4"), (1.0, "1:1"), (4 / 3, "4:3")]:
    ax_hist.axvline(ratio, color=INK, lw=0.8, ls=":")
    ax_hist.text(ratio, ax_hist.get_ylim()[1] * 0.97, name, ha="center", va="top", fontsize=8, color=INK)
ax_hist.set_xlabel("aspect ratio = width / height  (left of 1: portrait, right: landscape)")
ax_hist.set_ylabel("photos")
ax_hist.set_title("Aspect ratios")
fig.tight_layout()
display(fig)
plt.close(fig)

💡 The dots do not fill the plane: they sit on a few lines and clusters, because the photos share a common
longest side. The aspect histogram shows the same thing from the side: a landscape peak, a portrait peak and
a long tail of panoramas and tall crops. Nothing here is "wrong", but it means a model fed 224×224 squares
will see most of these photos squashed or cropped (the 🔀 Callout at the end of this section).

**Worked answer (Q4).** Not one size, and not a random cloud either: COCO downloaded its photos from Flickr
at one standard size, so the longest side is capped at the same value for almost every photo and the dots
line up along "width = cap" and "height = cap". A dataset from the park's own cameras would look different:
one dot per camera model (every frame of a camera has the same size), which is both tidier and a leakage
risk you will meet in section 6 (the size gives away the camera).

### Colour modes and colours

How a file stores colour is easy to forget and painful to discover during training: a single grayscale (`L`)
or `RGBA` file in a batch of `RGB` arrays crashes the data loader, or silently gets a channel duplicated or
dropped. The next cell counts the modes and draws the typical RGB histogram of each class.

In [ ]:
# Colour modes per class (always listing RGB / L / RGBA / CMYK), then the mean R, G, B histogram per class.
_modes = COLOUR_MODES + sorted(set(image_stats["mode"]) - set(COLOUR_MODES))
mode_by_class = pd.crosstab(image_stats["mode"], image_stats["class"])
mode_by_class = mode_by_class.reindex(index=_modes, columns=CLASSES, fill_value=0)
image_stats["looks_gray"] = (image_stats["mode"] == "L") | (image_stats["colourfulness"] < 2.0)
print("colour modes: " + " · ".join(f"{m} {int(mode_by_class.loc[m].sum())}" for m in _modes))
_n_gray = int(image_stats["looks_gray"].sum())
print(f"photos that look grayscale (mode L, or RGB with equal channels): {_n_gray}")

fig = plt.figure(figsize=(13, 4.6))
_gs = fig.add_gridspec(
    2, 4, width_ratios=[1.4, 1, 1, 1], hspace=0.65, wspace=0.3, left=0.05, right=0.99, top=0.86, bottom=0.12
)
ax_modes = fig.add_subplot(_gs[:, 0])
_bottom = np.zeros(len(_modes))
for c in CLASSES:
    ax_modes.bar(_modes, mode_by_class[c].to_numpy(), bottom=_bottom, color=CLASS_COLORS[c], label=c)
    _bottom += mode_by_class[c].to_numpy()
ax_modes.set_ylabel("photos")
ax_modes.set_title("Colour mode as stored in the file", fontsize=10)
ax_modes.legend(frameon=False, fontsize=8)

_edges = np.linspace(0, 256, HIST_BINS + 1)
_centres = (_edges[:-1] + _edges[1:]) / 2
for k, c in enumerate(CLASSES):
    ax = fig.add_subplot(_gs[k // 3, 1 + k % 3])
    _mean_hist = CHANNEL_HISTS[(image_stats["class"] == c).to_numpy()].mean(axis=0)
    for ch_idx, ch in enumerate("RGB"):
        ax.plot(_centres, _mean_hist[ch_idx], color=CHANNEL_COLORS[ch], lw=1.4, label=ch)
    ax.set_title(f"{c} ({int((image_stats['class'] == c).sum())} photos)", color=CLASS_COLORS[c], fontsize=10)
    ax.set_yticks([])
    ax.set_xlim(0, 255)
    ax.tick_params(labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor(CLASS_COLORS[c])
        spine.set_linewidth(1.6)
    if k == 0:
        ax.legend(frameon=False, fontsize=7, loc="upper left")
fig.text(0.67, 0.95, "Mean R, G, B histogram per class (pixel value 0–255)", ha="center", color=INK)
display(fig)
plt.close(fig)

💡 Read the histograms as *typical colours of a class*: a class whose curves differ from the others' (a
strong blue channel for a pond, high values across all three for a sandy savanna) is a class a model can
recognise from colour alone. That is a shortcut you should know about: a dog on green grass in the paddock
may be classified as "sheep" because the model learned "green = paddock = sheep".

### Brightness and the extremes

Brightness is the mean grey level of a photo (0 black, 255 white). Its distribution tells you whether the
dataset has night shots at all, and its extremes are the fastest route to broken or odd files.

In [ ]:
# Brightness per class as box plots, in class colours. The dotted lines mark "very dark" and "very bright".
fig, ax = plt.subplots(figsize=(9, 3.8))
_data = [image_stats.loc[image_stats["class"] == c, "brightness"] for c in CLASSES]
_box = ax.boxplot(_data, tick_labels=CLASSES, patch_artist=True, widths=0.6, medianprops={"color": INK})
for patch, c in zip(_box["boxes"], CLASSES, strict=True):
    patch.set_facecolor(CLASS_COLORS[c])
    patch.set_alpha(0.85)
for level in (40, 215):
    ax.axhline(level, color=INK, lw=0.8, ls=":")
ax.set_ylim(0, 255)
ax.set_ylabel("brightness (mean grey level)")
ax.set_title("Brightness per class")
fig.tight_layout()
display(fig)
plt.close(fig)

💡 Every class covers most of the range, so the dataset has dim and bright photos, but the medians sit
comfortably in the middle: these are daytime holiday photos. The park's cameras also run at dusk and at
night, which this data cannot teach.

Numbers point, eyes decide. `show_extremes(stat, n)` shows the `n` photos at both ends of any column of
`image_stats`, so you can see *what* a very dark, very wide or very grey photo actually is.

<table width="100%"><tr><td bgcolor="#FFF4D6"><font color="#7A4A00">
<b>✏️ Your turn · Look at the extremes</b><br>
Set <code>STAT_TO_SHOW</code> to one of <code>brightness, contrast, colourfulness, aspect, megapixels,
long_side</code> and <code>N_EXTREMES</code> to how many photos per end you want, then run the cell. Try
<code>"brightness"</code> first, then <code>"colourfulness"</code> (the low end is where grayscale pictures
hide) and <code>"aspect"</code>.
</font></td></tr></table>

In [ ]:
# ✏️ YOUR TURN
STAT_TO_SHOW = "brightness"
N_EXTREMES = 4

if not isinstance(N_EXTREMES, int) or N_EXTREMES < 1:
    raise SystemExit(f"N_EXTREMES must be a positive integer, not {N_EXTREMES!r}")
n_extremes = min(N_EXTREMES, 3) if SMOKE else N_EXTREMES  # smoke mode caps the gallery, same code (N-014)
show_extremes(STAT_TO_SHOW, n_extremes)

💡 The darkest photos are usually real (an animal in shade, a night shot with flash) and worth keeping: they
are the kind of frame the park's cameras produce at dusk. The brightest are often snow, sand or an
over-exposed sky. When one end shows a uniform grey rectangle, a thumbnail-sized file or a photo of a photo,
you have found a file for section 5's cleaning list.

### The detective beat: a cluster is a clue

A statistic that most photos share, and a few do not, is more than a curiosity. Photos from one pipeline
(one camera, one download script, one phone model) come out with the same size and mode. A **cluster of odd
sizes or modes therefore points to a different source**: another camera, another download, a batch someone
converted by hand. On the park's cameras you would see it as two size clusters (two camera models) or a
group of grayscale frames (the infrared night mode). Here we look for the same thing in COCO.

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
Look back at the scatter from Q4 before you run the next cell.
</font></td></tr></table>

**Q5.** 🟡 Which longest-side value do most photos share, and how many photos do you think do *not* share
it? Write down a guess, then run the cell: it counts them, lists their sizes and shows them.

In [ ]:
# Q5  The most common longest side, the photos that break the pattern, and the photos that look grayscale.
COMMON_LONG_SIDE = int(image_stats["long_side"].mode().iloc[0])
odd_size = image_stats[image_stats["long_side"] != COMMON_LONG_SIDE].sort_values("long_side")
gray_like = image_stats[image_stats["looks_gray"]]

print(
    f"{len(image_stats) - len(odd_size)} of {len(image_stats)} photos have a longest side of "
    f"{COMMON_LONG_SIDE} px; {len(odd_size)} do not."
)
if len(odd_size):
    _sizes = odd_size.apply(lambda r: f"{r['width']}×{r['height']}", axis=1).value_counts()
    print("odd sizes: " + " · ".join(f"{s} ×{n}" for s, n in _sizes.items()))
else:
    print("every photo in this sample went through the same resize step.")
print(f"{len(gray_like)} photos look grayscale.")

N_ODD = 6 if SMOKE else 12  # how many of the odd ones to show (N-014)
show_rows(odd_size.head(N_ODD), f"photos whose longest side is not {COMMON_LONG_SIDE} px")
show_rows(gray_like.head(N_ODD), "photos that look grayscale")

💡 The odd sizes are small, not large: nobody uploaded a 4000-pixel photo here. The photos that fall below
the common cap were *already* smaller than it when COCO fetched them, which is a hint about their origin
(old uploads, low-resolution cameras, screenshots). A grayscale photo in a colour dataset is a similar hint,
and both go on the list of things to decide about in section 5.

**Worked answer (Q5).** On the full Play Dataset almost every photo has a longest side of 640 px, because
COCO downloaded Flickr's "640 px" rendition, and only a handful fall below it (none may appear in a small
sample). The number itself matters less than the habit: whenever a column of `image_stats` has one dominant
value, ask "what made the others different?", because the answer is usually a different source, and a
different source usually means different labels, lighting or quality. On your own dataset (section 11) this
is the first question to ask about size and mode.

### Across tasks: what resizing does to boxes and masks

Everything above was about the photo. The park's second and third jobs (count, outline) attach **boxes** and
**masks** to the photo, and they are affected by the first thing every training pipeline does: resize.

<table width="100%"><tr><td bgcolor="#EDE7F6"><font color="#311B92">
<b>🔀 Across tasks · Resizing</b><br>
<ul>
<li><b>Classification:</b> squashing a 4:3 photo into a 224×224 square distorts aspect ratios. Usually
harmless for "which animal", because every photo is distorted the same way. Cropping or padding keeps the
shape at the cost of pixels or borders.</li>
<li><b>Detection:</b> box coordinates live in pixels, so a box must be rescaled with the image: x and width
by <code>new_width / width</code>, y and height by <code>new_height / height</code>. A forgotten rescale is
the classic "the boxes are all in the top-left corner" bug.</li>
<li><b>Segmentation:</b> a mask holds class ids, not colours. Resizing it with the usual smooth (bilinear)
filter invents in-between values that are no class at all; masks need <b>nearest-neighbour</b>
resizing.</li>
</ul>
</font></td></tr></table>

**Q6.** 🟡 A box is stored as `(x, y, w, h)` in pixels on a `W × H` photo, and the photo is resized to
224×224. Write the formula for the new box. Then predict: how many distinct values does the mask hold after
a nearest-neighbour resize, and after a bilinear one? Run the cell to check both.

In [ ]:
# Q6  One photo, one box and one mask, squeezed to 224×224: the box is rescaled; the mask is resized two ways.
TARGET = 224
_polygon = annotations["segmentation"].apply(lambda s: isinstance(s, list))
_ann = annotations[(annotations["iscrowd"] == 0) & _polygon].iloc[0]
_photo = images.loc[images["id"] == _ann["image_id"]].iloc[0]
_color = CLASS_COLORS[_ann["class"]]

with Image.open(_photo["path"]) as _im:
    original = _im.convert("RGB")
W, H = original.size
sx, sy = TARGET / W, TARGET / H
x, y, w, h = _ann["bbox"]
new_box = (x * sx, y * sy, w * sx, h * sy)

mask = Image.new("L", (W, H), 0)
for _poly in _ann["segmentation"]:
    ImageDraw.Draw(mask).polygon(list(zip(_poly[::2], _poly[1::2], strict=True)), fill=1)
squashed = original.resize((TARGET, TARGET), Image.BILINEAR)
mask_nearest = np.asarray(mask.resize((TARGET, TARGET), Image.NEAREST))
mask_bilinear = np.asarray(mask.point(lambda v: v * 255).resize((TARGET, TARGET), Image.BILINEAR)) / 255.0

print(f"photo {W}×{H} → {TARGET}×{TARGET}: scale x by {sx:.3f}, y by {sy:.3f}")
_old = ", ".join(f"{v:.0f}" for v in (x, y, w, h))
_new = ", ".join(f"{v:.1f}" for v in new_box)
print(f"box  (x, y, w, h) = ({_old})  →  ({_new})")
print(
    f"mask values: original {np.unique(np.asarray(mask)).tolist()} · "
    f"nearest {np.unique(mask_nearest).tolist()} · bilinear {len(np.unique(mask_bilinear))} distinct values"
)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.9))
axes[0].imshow(original)
axes[0].add_patch(Rectangle((x, y), w, h, fill=False, edgecolor=_color, lw=2))
axes[0].set_title(f"original {W}×{H} + box", fontsize=9)
axes[1].imshow(squashed)
axes[1].add_patch(Rectangle(new_box[:2], new_box[2], new_box[3], fill=False, edgecolor=_color, lw=2))
axes[1].set_title(f"resized {TARGET}×{TARGET} + rescaled box", fontsize=9)
axes[2].imshow(mask_nearest, cmap=ListedColormap(["white", _color]), vmin=0, vmax=1)
axes[2].set_title(f"mask, nearest: {len(np.unique(mask_nearest))} values", fontsize=9)
_soft_cmap = LinearSegmentedColormap.from_list("mask", ["white", _color])
axes[3].imshow(mask_bilinear, cmap=_soft_cmap, vmin=0, vmax=1)
axes[3].set_title(f"mask, bilinear: {len(np.unique(mask_bilinear))} values", fontsize=9)
for ax in axes:
    frame(ax, _color, width=2)
fig.suptitle(f"{_ann['class']}: what one resize does to the photo, its box and its mask", color=INK)
fig.tight_layout()
display(fig)
plt.close(fig)

💡 The box follows the photo as long as you scale x and y separately. The mask does not survive a smooth
resize: the bilinear version has a soft grey rim of values between 0 and 1, which are not "background" and
not "the animal". With six classes those in-between values would become ids of *other* animals.

**Worked answer (Q6).** New box = `(x · 224/W, y · 224/H, w · 224/W, h · 224/H)`; x-like values use the
horizontal factor, y-like values the vertical one, and on a landscape photo the two factors differ, so the
box is squashed exactly like the photo. Nearest-neighbour resizing keeps the two original values (0 and 1).
Bilinear resizing produces many distinct values along every edge of the animal, and the number grows with
the length of the boundary, so thin parts (legs, tails, a bird's beak) suffer most. That is why segmentation
pipelines resize images with a smooth filter and masks with nearest-neighbour.

**Takeaway.** `image_stats` costs seconds and already answers four of the checklist's questions (size,
shape, colour mode, colour) for any folder of images, labelled or not. Section 4 adds the labels and asks
the question every count depends on: how many photos of each class do we have, and does it matter?

## 4. Labels & class distribution

**Objective:** count the photos of each class, see why a photo with several animals is a labelling problem,
and understand why the dog alert cares about imbalance more than about accuracy.

Sections 2 and 3 looked at the photos. This section looks at the **labels**: the one word per photo that a
classifier will learn to say. Two questions decide whether the labels are usable at all: how many photos of
each class do we have, and is "one word per photo" even the right shape for this data? Karpathy lists the
things he looks for at this point:

<table width="100%"><tr><td bgcolor="#E8F5E9"><font color="#1B5E20">
<b>🧾 Recipe · Step 1, continued: imbalances and biases</b><br>
<i>"Luckily, your brain is pretty good at this. One time I discovered that the data contained duplicate
examples. Another time I found corrupted images / labels. I look for data imbalances and biases."</i><br>
Andrej Karpathy, <a href="https://karpathy.github.io/2019/04/25/recipe/">A Recipe for Training Neural
Networks</a> (2019).
</font></td></tr></table>

Duplicates and corrupted files are section 5's job. Here we take the last sentence: imbalances and biases
in the labels.

### Images per class

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
Predict before you run. Section 1's checklist already printed the per-class counts once; this time you will
see them as a bar chart, next to each class's share of the dataset.
</font></td></tr></table>

**Q7.** 🟢 Which class do you expect to have the most photos, which the fewest, and roughly how many times
larger is the biggest class than the smallest? Write down your guess, then run the cell.

In [ ]:
# Q7  Photos per class as bars (CLASS_COLORS), with each class's share of the dataset written on the bar.
images_per_class = {c: int((images["class"] == c).sum()) for c in CLASSES}
RESULTS["images_per_class"] = images_per_class

_total = sum(images_per_class.values())
_largest = max(images_per_class, key=images_per_class.get)
_smallest = min(images_per_class, key=images_per_class.get)
_ratio = images_per_class[_largest] / max(1, images_per_class[_smallest])
print(f"{_total} photos · largest class {_largest} ({images_per_class[_largest]}) · smallest {_smallest} "
      f"({images_per_class[_smallest]}) · ratio {_ratio:.1f}×")

fig, ax = plt.subplots(figsize=(9, 3.8))
_order = sorted(CLASSES, key=images_per_class.get, reverse=True)
ax.bar(_order, [images_per_class[c] for c in _order], color=[CLASS_COLORS[c] for c in _order])
for i, c in enumerate(_order):
    ax.text(i, images_per_class[c], f"{100 * images_per_class[c] / _total:.0f} %", ha="center", va="bottom",
            fontsize=9)
ax.set_ylabel("photos")
ax.set_title("Images per class (one label per photo)")
ax.set_ylim(0, max(images_per_class.values()) * 1.15)
fig.tight_layout()
display(fig)
plt.close(fig)

💡 The bars are not flat: the pet classes come first because people photograph their pets, and sheep come
last because few people photograph sheep. Nobody designed this distribution; it is what Flickr uploads look
like, and it has nothing to do with what the park's cameras will see (sheep all day, a dog once a month).

**Worked answer (Q7).** On the full Play Dataset cat and dog are the largest classes (capped at about 130
photos each) and sheep is the smallest (about 50), so the biggest class is roughly 2–3× the smallest. In the
small smoke sample the bars are flat, because that sample takes the same number of photos per class.
Whatever the sample, the habit is the same: read the *ratio* between the largest and the smallest class, and
ask whether the smallest class is the one your use case cares about. Here it is not sheep that matters, it
is dog, and dog is well represented in the photos while being the *rare* event in the paddock.

### Several animals in one photo: which label then?

`images` has one `class` per photo, and section 0 already showed that `n_animals` is often above 1. That is
fine as long as all the animals in a photo are the same class, and it is a real problem the moment they are
not. The Play Dataset was built by keeping only COCO photos with **exactly one animal class**, which is a
decision someone took *for* you. Check what the decision hides.

**Q8.** 🟡 What share of the photos do you think hold more than one animal, and which class do you expect to
be the most "crowded"? Then run the cell: it counts animals per photo and shows the most crowded photos.

In [ ]:
# Q8  Animals per photo, stacked by class, then a gallery of the photos with the most animals.
_max_n = int(images["n_animals"].max())
_several = images[images["n_animals"] > 1]
print(f"{len(_several)} of {len(images)} photos ({100 * len(_several) / len(images):.0f} %) hold more than "
      f"one animal; the most crowded photo has {_max_n}.")
print("mean animals per photo: " + " · ".join(
    f"{c} {images.loc[images['class'] == c, 'n_animals'].mean():.1f}" for c in CLASSES))

fig, ax = plt.subplots(figsize=(9, 3.8))
_bins = np.arange(0.5, min(_max_n, 10) + 1.5, 1)
_per_class = [images.loc[images["class"] == c, "n_animals"].clip(upper=10) for c in CLASSES]
ax.hist(_per_class, bins=_bins, stacked=True, color=[CLASS_COLORS[c] for c in CLASSES], label=CLASSES)
ax.set_xticks(range(1, min(_max_n, 10) + 1))
ax.set_xticklabels([str(i) if i < 10 else "10+" for i in range(1, min(_max_n, 10) + 1)])
ax.set_xlabel("animals in the photo")
ax.set_ylabel("photos")
ax.set_title("Animals per photo")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
display(fig)
plt.close(fig)

N_CROWDED = 3 if SMOKE else 6  # how many crowded photos to show (N-014)
_crowded = images.sort_values(["n_animals", "id"], ascending=[False, True]).head(N_CROWDED)
show_rows(_crowded, "the most crowded photos", lambda r: f"{r['n_animals']} × {r['class']}", cols=N_CROWDED)

💡 "One label per photo" works here only because every animal in a photo is the same species, and even then
the label says nothing about *how many*: a flock of thirty sheep and a single lamb are both "sheep". Look
at the crowded photos: some also hold people, fences or a second species that COCO does not label as an
animal class we kept, and the label is silent about all of it.

**Worked answer (Q8).** On the full Play Dataset about a third of the photos hold more than one animal, and
sheep and birds are the crowded classes (flocks), while cats and dogs are mostly alone. "Which label then?"
has three honest answers, and the park has to pick one before labelling a single frame:
- **one label, the main animal**: cheap, but the labeller decides what "main" means, and the model learns
  that fuzzy rule;
- **one label per animal class present** (multi-label): "sheep and dog" is exactly the paddock alert, but
  it needs a different loss and metric than plain classification;
- **one box per animal** (detection, section 8): answers "which" and "how many" at once, at a higher
  annotation cost (the 🔀 Callout below).

COCO's builders took the first answer *for the photos they kept*, and the Play Dataset went further by
dropping every photo with two classes. For the paddock camera, where the one case that matters is a dog
**among** sheep, that filter would delete exactly the frames the alert is for.

### Why the dog alert cares about imbalance

Imbalance is harmless in a bar chart and dangerous in a metric. The park's paddock camera sees sheep almost
all the time and a dog very rarely, and the usual score, accuracy, is blind to that.

<table width="100%"><tr><td bgcolor="#FDECEA"><font color="#8B1A10">
<b>⚠️ Pitfall · Accuracy hides a rare class</b><br>
Accuracy is the share of all photos labelled correctly, so it is dominated by the big classes. A model that
<b>never</b> predicts a rare class loses only that class's share of the points. If dogs are 2 % of the
paddock frames, "always say sheep" scores 98 % accuracy and misses every dog, which is the one event the
alert exists for. The number to watch for a rare class is its <b>recall</b>: of the real dogs, how many did
we catch? Section 7 turns this into the park's metric.
</font></td></tr></table>

No model is needed to feel this. The next cell builds the laziest classifier there is, "always predict the
biggest class", and scores it twice: on the Play Dataset and on a paddock day whose dog share you choose.

<table width="100%"><tr><td bgcolor="#FFF4D6"><font color="#7A4A00">
<b>✏️ Your turn · A paddock day</b><br>
<code>DOG_SHARE</code> is the fraction of the paddock camera's frames that show a dog (the rest show sheep).
Try <code>0.02</code>, then <code>0.2</code>, then <code>0.5</code>, and watch what happens to the accuracy
of "always sheep" and to its recall on dog.
</font></td></tr></table>


In [ ]:
# ✏️ YOUR TURN
DOG_SHARE = 0.02

if not isinstance(DOG_SHARE, (int, float)) or not 0 <= DOG_SHARE <= 1:
    raise SystemExit(f"DOG_SHARE must be a number between 0 and 1, not {DOG_SHARE!r}")

**Q9.** 🟡 Before you run: what accuracy does "always predict the biggest class" get on the Play Dataset,
and what is its recall on dog? And on a paddock day with the `DOG_SHARE` you set? Write both down, then run.

In [ ]:
# Q9  "Always the biggest class" on the Play Dataset: every photo gets the same label.
_majority = max(images_per_class, key=images_per_class.get)
_pred = np.full(len(images), _majority)
_true = images["class"].to_numpy()
acc_play = float((_pred == _true).mean())
recall_dog_play = float((_pred[_true == "dog"] == "dog").mean()) if (_true == "dog").any() else float("nan")
print(f"Play Dataset · 'always {_majority}': accuracy {100 * acc_play:.0f} %, recall on dog "
      f"{100 * recall_dog_play:.0f} %")

# A paddock day: PADDOCK_FRAMES frames, a share DOG_SHARE of them show a dog, the rest sheep.
PADDOCK_FRAMES = 200 if SMOKE else 1000
_n_dog = int(round(DOG_SHARE * PADDOCK_FRAMES))
_day = np.array(["dog"] * _n_dog + ["sheep"] * (PADDOCK_FRAMES - _n_dog))
_always_sheep = np.full(PADDOCK_FRAMES, "sheep")
acc_day = float((_always_sheep == _day).mean())
recall_dog_day = float((_always_sheep[_day == "dog"] == "dog").mean()) if _n_dog else float("nan")
print(f"Paddock day   · {_n_dog} dog frames in {PADDOCK_FRAMES}: 'always sheep' accuracy "
      f"{100 * acc_day:.0f} %, recall on dog {100 * recall_dog_day:.0f} %")

fig, ax = plt.subplots(figsize=(9, 3.4))
_shares = np.linspace(0, 0.5, 51)
_acc_curve = (1 - _shares) * 100
ax.plot(_shares * 100, _acc_curve, color=CLASS_COLORS["sheep"], lw=2, label="accuracy of 'always sheep'")
ax.plot(_shares * 100, np.zeros_like(_shares), color=CLASS_COLORS["dog"], lw=2, label="its recall on dog")
ax.axvline(DOG_SHARE * 100, color=INK, lw=0.8, ls=":")
ax.text(DOG_SHARE * 100, 50, f" DOG_SHARE = {DOG_SHARE:g}", color=INK, fontsize=9)
ax.set_xlabel("share of paddock frames that show a dog (%)")
ax.set_ylabel("score (%)")
ax.set_ylim(-5, 105)
ax.set_title("The laziest classifier on a paddock day")
ax.legend(frameon=False, fontsize=9, loc="center right")
fig.tight_layout()
display(fig)
plt.close(fig)

💡 Accuracy of the lazy classifier *is* the majority share: the rarer the dog, the better "always sheep"
looks, and its recall on dog stays at zero the whole way. A metric that rewards ignoring the event you built
the system for is the wrong metric.

**Worked answer (Q9).** On the Play Dataset "always the biggest class" scores the biggest class's share
(about 20 % on the full set, a sixth in the balanced smoke sample) and a recall on dog of 0 % unless dog
*is* the biggest class, in which case it scores 100 % on dog and 0 % on everything else. On a paddock day
with `DOG_SHARE = 0.02` it reaches 98 % accuracy with 0 % recall on dog. Two lessons: a balanced-looking
Play Dataset says nothing about the imbalance of the *deployment* (the paddock), and whatever the imbalance,
the score has to be computed on the class that matters. Training-time fixes for imbalance (class weights,
oversampling, a focal loss) are next week's topic; choosing the metric is section 7's.

### Across tasks: what a label costs

All of the above assumed "one word per photo". The park's other two jobs need richer labels, and richer
labels cost more to make. The next cell counts what the Play Dataset's annotations contain, in the crudest
unit there is: how many clicks a labeller had to make.

<table width="100%"><tr><td bgcolor="#EDE7F6"><font color="#311B92">
<b>🔀 Across tasks · One label, one box, every pixel</b><br>
<ul>
<li><b>Classification:</b> one label per image. One click (or one keystroke) per photo; seconds each. The
cheapest label, and the one that cannot say "how many" or "where".</li>
<li><b>Detection:</b> one box per animal. Two clicks per box, and a decision for every animal (is that blur
a bird?); tens of seconds per photo, more when the photo is crowded. Imbalance is now counted in
<i>boxes</i>, and the ranking of the classes changes (section 8).</li>
<li><b>Segmentation:</b> every pixel gets a class. A polygon of many vertices per animal; a minute or more
per animal, and the hardest to get right at thin parts. Imbalance is counted in <i>pixels</i>, and birds end
up with a few per cent of them (section 9).</li>
</ul>
The same photo therefore costs roughly 1 : 10 : 100 to label for the three tasks, which is why real datasets
have many more image labels than boxes, and many more boxes than masks.
</font></td></tr></table>

In [ ]:
# Clicks per task on the Play Dataset: 1 per image label, 2 per box, one per polygon vertex for masks.
_polygons = annotations["segmentation"].apply(lambda s: s if isinstance(s, list) else [])
_vertices = _polygons.apply(lambda polys: sum(len(p) // 2 for p in polys))
_n_crowd = int((annotations["iscrowd"] == 1).sum())
annotation_cost = pd.DataFrame(
    {
        "unit": ["image labels", "boxes", "mask vertices"],
        "count": [len(images), len(annotations), int(_vertices.sum())],
        "clicks": [len(images), 2 * len(annotations), int(_vertices.sum())],
    }
).set_index("unit")
annotation_cost["clicks per photo"] = (annotation_cost["clicks"] / len(images)).round(1)
_label_clicks = annotation_cost.loc["image labels", "clicks"]
annotation_cost["× classification"] = (annotation_cost["clicks"] / _label_clicks).round(1)
if _n_crowd:
    print(f"({_n_crowd} crowd regions are stored as pixel masks, not polygons, and count 0 vertices here.)")
display(annotation_cost)

💡 The same photos, the same animals, and the click count grows by an order of magnitude per task. When the
park asks for "a quick model that also counts and outlines", this table is the answer to "how quick".

**Takeaway.** Count the labels before you trust them: the ratio between the biggest and the smallest class,
the share of photos with several animals, and the share of the class your use case depends on. Then
remember that the deployment has its own imbalance, and that accuracy will happily ignore the rare class the
park built the camera for. Section 5 goes after the labels and photos that are *wrong*, not just rare.

## 5. Common dataset errors & cleaning

**Objective:** find duplicates, flipped and rotated copies, broken files and suspicious labels in a dataset,
and produce a cleaned table you can split and train on.

Sections 2 to 4 asked whether the data *fits* the task. This section asks whether the data is *wrong*: the
same photo stored twice, a copy that someone flipped in a viewer, a file that no longer opens, a label that
a tired annotator got wrong. None of these show up in a bar chart, and all of them bite later: a duplicate
that lands in both train and test makes the score lie, and a wrong label teaches the model a wrong rule.

<table width="100%"><tr><td bgcolor="#E8F5E9"><font color="#1B5E20">
<b>🧾 Recipe · Step 1, continued: the two bugs Karpathy names</b><br>
<i>"One time I discovered that the data contained duplicate examples. Another time I found corrupted
images / labels."</i><br>
Andrej Karpathy, <a href="https://karpathy.github.io/2019/04/25/recipe/">A Recipe for Training Neural
Networks</a> (2019).
</font></td></tr></table>

Two stories from Kaggle show how ordinary this is. In the <b>PetFinder.my Pawpularity</b> competition
(2021), teams found near-identical photos in the training set that carried *different* target scores, and
asked on the forum whether the same photos also sat in the hidden test set; the hosts answered that
duplicates between train and test had been removed, and left the pairs inside the training set as they
were, scores and all. In <b>Cassava Leaf Disease Classification</b> (2020), a competition known for its
noisy labels, teams reported a pair of identical leaf photos that carried two different disease labels.
Neither dataset was careless: both were built by people who cared, and both still shipped with duplicates.
Yours will too, unless you look.

### Exact duplicates: the same bytes twice

The cheapest check first. Two files that are byte-for-byte identical have the same **MD5 hash**, so one
pass over the files finds every exact copy, whatever its name.

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
COCO val2017 is a curated benchmark that thousands of papers evaluate on. Does that mean it has no exact
duplicates? Decide before you run.
</font></td></tr></table>

**Q10.** 🟢 How many byte-identical pairs do you expect in the Play Dataset: none, a handful, or many? Write
your guess, then run the next two cells.

In [ ]:
# @title Hash helpers (MD5, perceptual hash, flipped and rotated variants) { display-mode: "form" }
import hashlib
import shutil
from io import BytesIO

import imagehash
from PIL import Image, ImageOps

THUMB_FOR_HASH = 256  # perceptual hashes are computed on a thumbnail: fast, and the same for every file


def file_md5(path) -> str:
    """MD5 of the file's bytes: identical files share it, a one-pixel change breaks it."""
    return hashlib.md5(Path(path).read_bytes()).hexdigest()


def phash(image: Image.Image) -> imagehash.ImageHash:
    """Perceptual hash of a PIL image (64 bits). `h1 - h2` is the number of bits that differ (0–64)."""
    return imagehash.phash(image)


def variants(image: Image.Image) -> dict[str, Image.Image]:
    """The image as stored plus its mirrored, flipped and rotated versions, keyed by what was done."""
    return {
        "original": image,
        "flip_lr": ImageOps.mirror(image),
        "flip_tb": ImageOps.flip(image),
        "rot90": image.transpose(Image.Transpose.ROTATE_90),
        "rot180": image.transpose(Image.Transpose.ROTATE_180),
        "rot270": image.transpose(Image.Transpose.ROTATE_270),
    }


def hash_bits(hashes: list[imagehash.ImageHash]) -> np.ndarray:
    """Stack ImageHash objects into an (n, 64) boolean matrix, so distances can be computed in one go."""
    return np.array([h.hash.flatten() for h in hashes], dtype=bool)


def hash_distances(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """(len(a), len(b)) matrix of Hamming distances between two (n, 64) hash matrices."""
    return (a[:, None, :] != b[None, :, :]).sum(axis=-1)


def exact_duplicate_groups(paths: list) -> list[list]:
    """Groups of paths whose bytes are identical (only groups with 2+ members), in first-seen order."""
    by_md5: dict[str, list] = {}
    for p in paths:
        by_md5.setdefault(file_md5(p), []).append(p)
    return [group for group in by_md5.values() if len(group) > 1]


def is_readable(path) -> bool:
    """True if PIL can open and fully decode the file."""
    try:
        with Image.open(path) as im:
            im.draft("RGB", (THUMB_FOR_HASH, THUMB_FOR_HASH))  # a reduced decode still reads every byte
            im.load()
        return True
    except Exception:
        return False

In [ ]:
# Q10  MD5 of every file in the Play Dataset; identical bytes share one hash.
exact_groups = exact_duplicate_groups(images["path"].tolist())
print(f"{len(images)} files · {len(exact_groups)} groups of byte-identical files")
for _group in exact_groups[:5]:
    print("  ", " = ".join(Path(p).name for p in _group))

💡 Whatever the count, this check cost one line and one pass over the files; run it on every folder you are
handed, because exact copies are the duplicates that slip through most often (a batch uploaded twice, a
"final" and a "final_v2" export of the same card).

**Worked answer (Q10).** COCO val2017 was deduplicated by its authors, so on the Play Dataset the expected
answer is none or close to it. That is the exception, not the rule: the park's cameras, like any camera
upload, will produce exact copies the first time someone re-exports a memory card. And MD5 is blind to
everything but identical bytes: re-save the same photo at a different JPEG quality and the hash changes
completely. For that we need a hash that looks at the picture, not the file.

### Near-duplicates: the same picture, different bytes

<table width="100%"><tr><td bgcolor="#E3F2FD"><font color="#0D47A1">
<b>📘 Theory · Perceptual hashing</b><br>
A <b>perceptual hash</b> (here pHash, from the <code>imagehash</code> package) shrinks the photo to a 32×32
grey thumbnail, takes its low-frequency content (a DCT, as in JPEG), and writes 64 bits saying whether each
coefficient is above or below the median. Two photos that <i>look</i> the same get almost the same bits
even when their files differ (resized, re-compressed, slightly cropped). The <b>Hamming distance</b>
between two hashes is the number of bits that differ: 0 means identical hashes, about 32 means unrelated.
A threshold on that distance decides what counts as "the same picture", and that threshold is a choice.
</font></td></tr></table>

<table width="100%"><tr><td bgcolor="#FFF4D6"><font color="#7A4A00">
<b>✏️ Your turn · The duplicate threshold</b><br>
<code>HASH_THRESHOLD</code> is the largest Hamming distance (0–64) at which two photos count as
near-duplicates. Start with the default, look at the closest pairs the next cell shows, then try a strict
<code>2</code> and a loose <code>14</code> and watch what enters the list.
</font></td></tr></table>

In [ ]:
# ✏️ YOUR TURN
HASH_THRESHOLD = 8

if not isinstance(HASH_THRESHOLD, int) or not 0 <= HASH_THRESHOLD <= 64:
    raise SystemExit(f"HASH_THRESHOLD must be a whole number between 0 and 64, not {HASH_THRESHOLD!r}")

**Q11.** 🟡 With your threshold, how many pairs of the Play Dataset do you expect to be flagged, and will
the closest pairs be true copies or just two photos of the same kind of scene? Guess, then run.

In [ ]:
# Q11  pHash of every photo (and of its flipped/rotated variants, used below), then the closest pairs.
VARIANT_NAMES = list(variants(Image.new("RGB", (4, 4))))
_per_variant: dict[str, list] = {v: [] for v in VARIANT_NAMES}
for _p in images["path"]:
    _thumb = load_thumb(_p, THUMB_FOR_HASH)
    for _v, _im in variants(_thumb).items():
        _per_variant[_v].append(phash(_im))
VARIANT_HASHES = {v: hash_bits(hs) for v, hs in _per_variant.items()}  # each (n_images, 64)
HASHES = VARIANT_HASHES["original"]

_dist = hash_distances(HASHES, HASHES)
np.fill_diagonal(_dist, 99)
nearest_distance = _dist.min(axis=1)  # each photo's distance to its closest other photo
_i, _j = np.where(np.triu(_dist <= HASH_THRESHOLD, k=1))
near_pairs = pd.DataFrame({"a": images["id"].to_numpy()[_i], "b": images["id"].to_numpy()[_j],
                           "distance": _dist[_i, _j]}).sort_values("distance").reset_index(drop=True)
print(f"pairs within {HASH_THRESHOLD} bits: {len(near_pairs)} · median nearest-neighbour distance: "
      f"{np.median(nearest_distance):.0f} bits")

fig, ax = plt.subplots(figsize=(9, 3.4))
_bins = np.arange(-0.5, 41, 2)
ax.hist([nearest_distance[(images["class"] == c).to_numpy()] for c in CLASSES], bins=_bins, stacked=True,
        color=[CLASS_COLORS[c] for c in CLASSES], label=CLASSES)
ax.axvline(HASH_THRESHOLD + 0.5, color=INK, ls=":", lw=1)
ax.text(HASH_THRESHOLD + 1, ax.get_ylim()[1] * 0.9, f"HASH_THRESHOLD = {HASH_THRESHOLD}", fontsize=9)
ax.set_xlabel("Hamming distance to the closest other photo (bits)")
ax.set_ylabel("photos")
ax.set_title("How close is each photo's nearest neighbour?")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
display(fig)
plt.close(fig)

# The closest pairs, side by side, whether or not they pass the threshold: this is what "near" looks like.
N_PAIRS_TO_SHOW = 2 if SMOKE else 4
_upper = np.where(np.triu(np.ones_like(_dist, dtype=bool), k=1), _dist, 99)  # each pair once
_flat = np.argsort(_upper, axis=None)[:N_PAIRS_TO_SHOW]
_closest = list(zip(*np.unravel_index(_flat, _dist.shape), strict=True))
if len(near_pairs):
    display(near_pairs.head(10))
_rows = pd.concat([image_stats.iloc[[a, b]] for a, b in _closest])
_dist_of = {row_id: _dist[a, b] for a, b in _closest for row_id in images["id"].iloc[[a, b]]}
show_rows(_rows, "closest pairs by perceptual hash (each pair side by side)",
          lambda r: f"{r['class']} · {_dist_of[r['id']]} bits apart", cols=2)

💡 The histogram sits far to the right of the threshold for almost every photo: unrelated pictures are
about 30 bits apart, and the threshold only matters for the thin left tail. Look at the pairs: at a few bits
you see the same scene; at 10 or more you see two different photos that merely share a layout.

**Worked answer (Q11).** On COCO most of the closest pairs are different photos of a similar scene (two
zebras on grass, a cat on a sofa) at distances above the default threshold, so the flagged list is short.
Short, not empty: on the full Play Dataset the default threshold catches one true pair, two frames of the
same tabby cat in front of the same brick wall, looking away in one and at the camera in the other. A
curated benchmark, and still a burst that slipped through. (The small smoke sample may not contain it.)
With a loose threshold the list fills up with photos that are *not* copies. That is the trade-off: a strict
threshold misses re-compressed copies and bursts, a loose one deletes real data. Keep the default for now
and read the list by eye before deleting anything.

<table width="100%"><tr><td bgcolor="#FDECEA"><font color="#8B1A10">
<b>⚠️ Pitfall · A hash distance is not a similarity score</b><br>
Two burst frames of the same sheep can be 12 bits apart and two unrelated photos 10. The threshold only
separates "same picture, different file" from everything else; it does not rank how alike two photos are.
And a plain hash is <b>orientation-sensitive</b>: mirror a photo and its hash changes by dozens of bits,
so a flipped copy looks like a new photo. The next part fixes that.
</font></td></tr></table>

### A memory card from the park: the guided hunt

The park's intern hands you a memory card. Some files on it are the same photo twice: the camera software
saved a copy, someone rotated a photo in a viewer and saved it, a second version was exported at lower
quality for an e-mail. The next cell writes that card into `data/cache/hunt5/` (seeded, so everyone gets
the same one) with neutral file names. **The answer key is printed after the hunt, not before.**

In [ ]:
# @title Plant the memory card (seeded, at runtime; nothing is redistributed) { display-mode: "form" }
HUNT_DIR = DATA_DIR / "cache" / "hunt5"
N_CARD_ORIGINALS = 12 if SMOKE else 40  # distinct photos on the card (N-014: fewer in smoke mode)
PLANT_KINDS = ["exact", "exact", "flipped", "rotated", "recompressed"]  # ≥ 4 planted copies, always present


def plant_hunt5(dest: Path, n_originals: int, seed: int) -> tuple[dict[str, Path], list[dict]]:
    """Write a 'memory card': n_originals distinct photos plus one planted copy per entry of PLANT_KINDS.

    Returns (files, key): files maps the card's neutral file names to paths; key lists the planted copies as
    {"copy": name, "source": name, "kind": kind}. Deterministic for a given seed.
    """
    rng = random.Random(seed)
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True)
    chosen = images.sample(n=min(n_originals, len(images)), random_state=seed).sort_values("id")
    sources = [Path(p) for p in chosen["path"]]
    # (bytes, kind or None for an original, index of the source photo)
    entries: list[tuple[bytes, str | None, int]] = [(s.read_bytes(), None, i) for i, s in enumerate(sources)]
    planted: list[tuple[str, int]] = []
    source_idx = rng.sample(range(len(sources)), len(PLANT_KINDS))
    for kind, i in zip(PLANT_KINDS, source_idx, strict=True):
        with Image.open(sources[i]) as im:
            rgb = im.convert("RGB")
        buf = BytesIO()
        if kind == "exact":
            data = sources[i].read_bytes()
        elif kind == "flipped":
            ImageOps.mirror(rgb).save(buf, "JPEG", quality=92)
            data = buf.getvalue()
        elif kind == "rotated":
            rgb.transpose(Image.Transpose.ROTATE_90).save(buf, "JPEG", quality=92)
            data = buf.getvalue()
        else:  # recompressed: smaller and at a lower JPEG quality
            small = rgb.resize((max(8, int(rgb.width * 0.7)), max(8, int(rgb.height * 0.7))))
            small.save(buf, "JPEG", quality=55)
            data = buf.getvalue()
        entries.append((data, kind, i))
        planted.append((kind, i))
    order = list(range(len(entries)))
    rng.shuffle(order)
    names = {idx: f"card_{k:03d}.jpg" for k, idx in enumerate(order)}
    files, key = {}, []
    for idx, (data, kind, i) in enumerate(entries):
        path = dest / names[idx]
        path.write_bytes(data)
        files[names[idx]] = path
        if kind is not None:
            key.append({"copy": names[idx], "source": names[i], "kind": kind})
    return dict(sorted(files.items())), key

In [ ]:
hunt_files, HUNT5_KEY = plant_hunt5(HUNT_DIR, N_CARD_ORIGINALS, SEED)
hunt_images = {name: load_thumb(path, THUMB_FOR_HASH) for name, path in hunt_files.items()}
print(f"{len(hunt_files)} files on the card in {HUNT_DIR}/ · {len(HUNT5_KEY)} of them are copies of another "
      f"file on the card")

**Q12.** 🟡 The shoot-out. Two detectors run on the card: plain hashing (MD5 plus pHash with your threshold)
and *variant* hashing, which also hashes each photo's mirrored, flipped and rotated versions. Predict which
kinds of copy each one catches, then run.

In [ ]:
# @title Copy finder: plain hashing vs variant hashing { display-mode: "form" }
KIND_OF_VARIANT = {"original": "recompressed", "flip_lr": "flipped", "flip_tb": "flipped",
                   "rot90": "rotated", "rot180": "rotated", "rot270": "rotated"}


def find_copies(files: dict[str, Path], threshold: int, use_variants: bool) -> pd.DataFrame:
    """Pairs of files on a card that are copies of each other: columns a, b, kind, distance.

    kind is "exact" (same bytes), "recompressed" (hashes within `threshold`), or, with use_variants=True,
    "flipped" / "rotated" (a flipped or rotated variant of a hashes within `threshold` of b).
    """
    names = list(files)
    md5s = {n: file_md5(p) for n, p in files.items()}
    thumbs = {n: load_thumb(p, THUMB_FOR_HASH) for n, p in files.items()}
    plain = hash_bits([phash(thumbs[n]) for n in names])
    variant_names = VARIANT_NAMES if use_variants else ["original"]
    # distances[v][i, j]: hash of variant v of file i vs the plain hash of file j
    distances = {v: hash_distances(hash_bits([phash(variants(thumbs[n])[v]) for n in names]), plain)
                 for v in variant_names}
    rows = []
    for i, a in enumerate(names):
        for j in range(i + 1, len(names)):
            b = names[j]
            if md5s[a] == md5s[b]:
                rows.append({"a": a, "b": b, "kind": "exact", "distance": 0})
                continue
            best_v = min(variant_names, key=lambda v: distances[v][i, j])
            best_d = int(distances[best_v][i, j])
            if best_d <= threshold:
                rows.append({"a": a, "b": b, "kind": KIND_OF_VARIANT[best_v], "distance": best_d})
    return pd.DataFrame(rows, columns=["a", "b", "kind", "distance"])


def score_hunt(found: pd.DataFrame, key: list[dict]) -> int:
    """How many planted copies in `key` appear as a pair (either order) in `found`."""
    pairs = {frozenset((r["a"], r["b"])) for _, r in found.iterrows()}
    return sum(frozenset((k["copy"], k["source"])) in pairs for k in key)

In [ ]:
# Q12  Plain hashing first, then variant hashing, on the same card and the same threshold.
plain_found = find_copies(hunt_files, HASH_THRESHOLD, use_variants=False)
variant_found = find_copies(hunt_files, HASH_THRESHOLD, use_variants=True)
print(f"plain hashing   : {len(plain_found)} pairs → {dict(Counter(plain_found['kind']))}")
print(f"variant hashing : {len(variant_found)} pairs → {dict(Counter(variant_found['kind']))}")
display(variant_found)

💡 The two detectors agree on the exact and re-compressed copies and disagree on the rest: mirroring or
rotating a photo moves its hash by dozens of bits, so plain hashing files it as a new photo, while hashing
the variants brings it back within a few bits of its source.

**Worked answer (Q12).** Plain hashing catches the byte-identical copies (MD5) and the re-compressed one
(pHash is built to survive re-compression and resizing). It misses the flipped and the rotated copy, which
variant hashing catches because one of the six variants of the source now hashes like the copy. The price
is six hashes per photo instead of one, which is still seconds for thousands of photos.

### ✍️ Write the flip detector yourself

<table width="100%"><tr><td bgcolor="#FFF4D6"><font color="#7A4A00">
<b>✏️ Your turn · <code>find_flipped_copies</code></b><br>
Fill in the function below: given one photo and a dictionary of candidate photos, return the names of the
candidates that are a flipped or rotated copy of it. You have <code>variants(image)</code>,
<code>phash(image)</code>, <code>h1 - h2</code> for the distance and <code>HASH_THRESHOLD</code>. The
<code>use()</code> line checks your answer against the reference on the card and continues with the
reference either way.
</font></td></tr></table>

In [ ]:
# ✍️ YOUR CODE
def find_flipped_copies(image, candidates):
    """Return the names of the candidates that are a flipped or rotated copy of `image`.

    image: a PIL image. candidates: dict {name: PIL image} (the image itself is not among them).
    Returns a sorted list of names whose pHash is within HASH_THRESHOLD bits of the pHash of one of
    image's flipped or rotated variants (every variant except "original").
    """
    ...


def _ref_find_flipped_copies(image, candidates):
    flipped_hashes = [phash(im) for name, im in variants(image).items() if name != "original"]
    found = []
    for name, cand in candidates.items():
        h = phash(cand)
        if min(h - fh for fh in flipped_hashes) <= HASH_THRESHOLD:
            found.append(name)
    return sorted(found)


_flip_source = next(k["source"] for k in HUNT5_KEY if k["kind"] == "flipped")
_others = {name: im for name, im in hunt_images.items() if name != _flip_source}
flipped_copies = use(find_flipped_copies, hunt_images[_flip_source], _others,
                     reference=_ref_find_flipped_copies)
print(f"flipped/rotated copies of {_flip_source}: {flipped_copies}")

### The hunt's score and the answer key

In [ ]:
# Score the reference detector against the planted key, then show every planted pair side by side.
hunt5_found = score_hunt(variant_found, HUNT5_KEY)
RESULTS["hunt5"] = {"planted": len(HUNT5_KEY), "found": int(hunt5_found),
                    "kinds": dict(Counter(k["kind"] for k in HUNT5_KEY))}
print(f"planted copies found by variant hashing: {hunt5_found} of {len(HUNT5_KEY)}")
print("answer key:")
for _k in HUNT5_KEY:
    print(f"  {_k['copy']}: {_k['kind']} copy of {_k['source']}")

fig, axes = plt.subplots(2, len(HUNT5_KEY), figsize=(2.4 * len(HUNT5_KEY), 5.2), squeeze=False)
for _col, _k in enumerate(HUNT5_KEY):
    for _row, _name in enumerate((_k["source"], _k["copy"])):
        ax = axes[_row, _col]
        ax.imshow(hunt_images[_name])
        frame(ax, INK if _row == 0 else CLASS_COLORS["dog"])
        ax.set_title(f"{_name}\n{'source' if _row == 0 else _k['kind'] + ' copy'}", fontsize=8)
fig.suptitle("the planted copies (top: source, bottom: copy)", color=INK)
fig.tight_layout()
display(fig)
plt.close(fig)

💡 Every planted copy is a pair the detector reported, and the pairs it reported are only the planted ones:
that is the result to insist on with your own data too, a detector that finds everything you planted and
nothing else. Before you trust it on a real folder, plant a few copies and check it the same way.

### Broken and odd files

A file can be in the table and still be useless: truncated during download, a grayscale or CMYK scan among
RGB photos, a PNG with an alpha channel, a thumbnail of 40 pixels or a 50-megapixel panorama. The reader
will either crash on it or silently feed the model something strange. Section 3 already measured the
modes and sizes; this cell adds "does it even open?" and lists everything odd in one table.

In [ ]:
# Flags per file: unreadable, non-RGB mode, extreme size. Reading every file fully takes a few seconds.
EXTREME_SIDE_PX = (64, 6000)  # longest side below / above this is "extreme"


def flag_odd_files(stats: pd.DataFrame) -> pd.DataFrame:
    """One row per flagged file of `stats` (rows of image_stats): id, class, flags. Empty if all is well."""
    rows = []
    for _, r in stats.iterrows():
        flags = []
        if not is_readable(r["path"]):
            flags.append("unreadable")
        if r["mode"] != "RGB":
            flags.append(f"mode {r['mode']}")
        if r["long_side"] < EXTREME_SIDE_PX[0] or r["long_side"] > EXTREME_SIDE_PX[1]:
            flags.append(f"extreme size {r['width']}×{r['height']}")
        if flags:
            rows.append({"id": r["id"], "class": r["class"], "flags": ", ".join(flags)})
    return pd.DataFrame(rows, columns=["id", "class", "flags"])


odd_files = flag_odd_files(image_stats)
print(f"{len(odd_files)} of {len(image_stats)} files carry a flag")
display(odd_files.head(10))

# A broken file for comparison: the first half of a JPEG. `is_readable` catches it; a plain `open` would not.
_half = Path(images["path"].iloc[0]).read_bytes()
_broken = HUNT_DIR / "broken_demo.jpg"
_broken.write_bytes(_half[: len(_half) // 2])
print(f"a truncated copy of {Path(images['path'].iloc[0]).name}: readable = {is_readable(_broken)}")

💡 Flags are not verdicts. A grayscale photo may be a perfectly good night-camera frame, and a large
panorama may be the park's best shot of the whole paddock; the table says "look at me", and a human decides
whether to convert, resize or drop. Only "unreadable" is dropped without discussion.

### Suspicious labels: each class's least typical photos

A wrong label cannot be found by hashing. What can be done without a model is to sort each class by how
*unlike the rest of its class* a photo is, using the colour histograms of section 3, and hand the top of
each list to a human. Most of them will be fine (an unusual but correct photo); the wrong ones, if any, are
far more likely to be here than in a random sample.

In [ ]:
# Each photo's distance to its own class's mean colour histogram; the farthest few per class go to a human.
N_ATYPICAL = 2 if SMOKE else 4
_features = CHANNEL_HISTS.reshape(len(image_stats), -1)
_class_mean = {c: _features[(image_stats["class"] == c).to_numpy()].mean(axis=0) for c in CLASSES}
image_stats["atypicality"] = [float(np.abs(f - _class_mean[c]).sum())
                              for f, c in zip(_features, image_stats["class"], strict=True)]

fig, ax = plt.subplots(figsize=(9, 3.4))
for k, c in enumerate(CLASSES):
    _vals = image_stats.loc[image_stats["class"] == c, "atypicality"].to_numpy()
    _jitter = np.random.RandomState(SEED).uniform(-0.25, 0.25, len(_vals))
    ax.scatter(k + _jitter, _vals, s=14, color=CLASS_COLORS[c], alpha=0.8)
ax.set_xticks(range(len(CLASSES)))
ax.set_xticklabels(CLASSES)
ax.set_ylabel("distance to the class's mean histogram")
ax.set_title("How typical is each photo of its class? (high = look at me)")
fig.tight_layout()
display(fig)
plt.close(fig)

least_typical = (image_stats.sort_values("atypicality", ascending=False).groupby("class", sort=False)
                 .head(N_ATYPICAL).sort_values(["class", "atypicality"], ascending=[True, False]))
show_rows(least_typical, "least typical photos per class: does the label fit?",
          lambda r: f"{r['class']} · {r['atypicality']:.1f}", cols=N_ATYPICAL * 2)

**Q13.** 🟡 Look at the gallery. For each photo, is the label right? Is the animal even visible? Note the
ones you would send back to the annotator; the next section's worked answer says what you typically find.

💡 Colour cannot tell a cat from a dog, so this list finds photos that are *unusual*, not photos that are
*wrong*: a black cat on white snow, a horse photographed at night. That is still useful, because an
unusual photo is where a tired annotator slips, and because a model will struggle on it either way.

**Worked answer (Q13).** On the Play Dataset most of the least typical photos are correctly labelled but
odd: close-ups, night shots, an animal that fills a tenth of the frame, a person holding the animal. Real
label errors in COCO val are rare at the image level (they are more common in the masks, section 9), so an
honest answer is "nothing to send back, two to keep an eye on". On a crowd-sourced dataset, the same list is
where the wrong labels live. The 🔴 part below replaces colour with a learned representation, which does
tell a cat from a dog.

### 🔴 Optional: model-based search on precomputed embeddings

A hash sees pixels; an **embedding** (the feature vector a pretrained network computes for a photo) sees
content, so two photos of the same dog from different angles land close together, and a photo whose
embedding sits far from every other photo of its class is the best candidate for a wrong label. No model
runs here: the vectors are read from a precomputed file if it is available. When it is not, the cell says so
and runs the same search on the colour histograms of section 3 as a stand-in, so you can see the mechanics.

In [ ]:
# @title Embedding helpers (precomputed vectors, or colour histograms as a stand-in) { display-mode: "form" }
EMBEDDINGS_NAME = "coco_animals_embeddings.npz"  # npz keys: ids (n,), embeddings (n, d); matched by image id


def load_embeddings() -> tuple[np.ndarray, str]:
    """(vectors aligned with `images`, source). Source is "precomputed" or "colour histograms (stand-in)".

    Reads data/coco_animals_embeddings.npz, fetching it from DATA_BASE_URL outside smoke mode. Never runs a
    model. Falls back to the colour histograms when the file is missing or does not cover every image.
    """
    path = DATA_DIR / EMBEDDINGS_NAME
    if not path.exists() and not SMOKE:
        download_file(DATA_BASE_URL + EMBEDDINGS_NAME, path, headers=_github_headers())
    if path.exists():
        try:
            z = np.load(path)
            by_id = dict(zip(z["ids"].tolist(), z["embeddings"], strict=True))
            if all(i in by_id for i in images["id"]):
                return np.stack([by_id[i] for i in images["id"]]).astype(np.float32), "precomputed"
        except Exception:
            pass
    return CHANNEL_HISTS.reshape(len(images), -1).astype(np.float32), "colour histograms (stand-in)"


def cosine_similarity_matrix(vectors: np.ndarray) -> np.ndarray:
    """Cosine similarity between every pair of rows (after centring the columns)."""
    centred = vectors - vectors.mean(axis=0)
    norm = np.linalg.norm(centred, axis=1, keepdims=True) + 1e-9
    unit = centred / norm
    return unit @ unit.T

**Q14.** 🔴 Will the embedding search's closest pairs be the same pairs the hash found, or different ones?
And will its least typical photos per class overlap with the colour-based list? Guess, then run.

In [ ]:
# Q14  Nearest pairs and per-class outliers in embedding space (or in the stand-in space).
embeddings, embedding_source = load_embeddings()
print(f"vectors: {embeddings.shape[0]} × {embeddings.shape[1]} from {embedding_source}")
RESULTS["embedding_source"] = embedding_source

_sim = cosine_similarity_matrix(embeddings)
np.fill_diagonal(_sim, -1)
_i, _j = np.unravel_index(np.argsort(-np.triu(_sim, k=1), axis=None)[:N_PAIRS_TO_SHOW], _sim.shape)
_rows = pd.concat([image_stats.iloc[[a, b]] for a, b in zip(_i, _j, strict=True)])
_sim_of = {row_id: _sim[a, b] for a, b in zip(_i, _j, strict=True) for row_id in images["id"].iloc[[a, b]]}
show_rows(_rows, f"closest pairs by {embedding_source}",
          lambda r: f"{r['class']} · sim {_sim_of[r['id']]:.2f}", cols=2)

_class_arr = image_stats["class"].to_numpy()
image_stats["class_similarity"] = [float(_sim[k, (_class_arr == c) & (np.arange(len(_sim)) != k)].mean())
                                   for k, c in enumerate(_class_arr)]
embedding_outliers = (image_stats.sort_values("class_similarity").groupby("class", sort=False)
                      .head(N_ATYPICAL).sort_values(["class", "class_similarity"]))
show_rows(embedding_outliers, f"least typical per class by {embedding_source}",
          lambda r: f"{r['class']} · sim {r['class_similarity']:.2f}", cols=N_ATYPICAL * 2)
_overlap = len(set(embedding_outliers["id"]) & set(least_typical["id"]))
print(f"{_overlap} of {len(embedding_outliers)} embedding outliers are also in the colour-based list")

💡 With real embeddings the closest pairs are *semantic* near-duplicates (the same dog, another angle),
which a hash cannot find, and the outliers are photos whose content does not match the label. With the
stand-in they are colour matches, which is why the cell says which one you are looking at.

**Worked answer (Q14).** The hash finds copies of the same *picture*; the embedding finds photos of the same
*thing*. The two lists overlap on exact and near copies and differ everywhere else, and the overlap between
the two outlier lists is partial at best: colour and content are different notions of "typical". To get
real embeddings, compute them once on a GPU (any pretrained image model), save `ids` and `embeddings` to
`data/coco_animals_embeddings.npz`, and this cell picks them up.

### Clean the dataset

Time to act. The rule applied below is the one most teams use: keep the first photo of every duplicate
group (exact, near, flipped or rotated), drop every unreadable file, and keep everything else with its
flag so a human can still decide. The result is `clean_images`, the table the rest of the notebook uses.

<table width="100%"><tr><td bgcolor="#FDECEA"><font color="#8B1A10">
<b>⚠️ Pitfall · Split after cleaning, not before</b><br>
If you split first and deduplicate each half on its own, a pair with one photo in train and one in test
survives, and the test score reports how well the model <i>memorised</i> that photo. Deduplicate the whole
table, then split (section 6).
</font></td></tr></table>

In [ ]:
# Duplicate groups on the Play Dataset with variant hashing, then the cleaned table and its summary line.
_ids = images["id"].to_numpy()
_drop: dict[int, str] = {}
for _group in exact_groups:  # exact: keep the first file of each group
    _group_ids = [int(images.loc[images["path"] == str(p), "id"].iloc[0]) for p in _group]
    for _gid in sorted(_group_ids)[1:]:
        _drop[_gid] = "exact duplicate"
_best = np.full(_dist.shape, 99)
for _v in VARIANT_NAMES:  # variant hashing across the whole Play Dataset: min distance over the six variants
    _best = np.minimum(_best, hash_distances(VARIANT_HASHES[_v], HASHES))
np.fill_diagonal(_best, 99)
_pi, _pj = np.where(np.triu(np.minimum(_best, _best.T) <= HASH_THRESHOLD, k=1))
for _a, _b in zip(_pi, _pj, strict=True):  # near/flipped/rotated: keep the lower id
    _drop.setdefault(int(max(_ids[_a], _ids[_b])), "near-duplicate (incl. flipped/rotated)")
for _, _r in odd_files.iterrows():
    if "unreadable" in _r["flags"]:
        _drop[int(_r["id"])] = "unreadable"

clean_images = images[~images["id"].isin(_drop)].copy()
clean_images["flags"] = clean_images["id"].map(odd_files.set_index("id")["flags"]).fillna("")
removed = pd.DataFrame({"id": list(_drop), "reason": list(_drop.values())})
RESULTS["n_clean"] = int(len(clean_images))
RESULTS["n_removed"] = int(len(removed))
_n_flagged = int((clean_images["flags"] != "").sum())
print(f"Play Dataset: removed {len(removed)}, kept {len(clean_images)} "
      f"(reasons: {dict(Counter(removed['reason']))}; {_n_flagged} kept with a flag)")

# The same rule on the memory card: one file per duplicate group.
_card_drop = set()
for _, _r in variant_found.sort_values(["a", "b"]).iterrows():
    if _r["a"] not in _card_drop:
        _card_drop.add(_r["b"])
print(f"memory card : removed {len(_card_drop)}, kept {len(hunt_files) - len(_card_drop)}")

💡 On a curated benchmark the "removed" number is small and the lesson is in the tooling; on the park's own
uploads the same six cells will be the difference between a test score you can trust and one you cannot.

<table width="100%"><tr><td bgcolor="#EDE7F6"><font color="#311B92">
<b>🔀 Across tasks · The same errors, three shapes</b><br>
<ul>
<li><b>Classification:</b> a duplicate pair across train and test leaks one label; a wrong label is one
word to fix.</li>
<li><b>Detection:</b> a duplicated photo duplicates <i>all</i> its boxes, and a flipped copy whose boxes
were not mirrored (<code>x' = W − x − w</code>) is worse than no copy at all: the boxes now point at
grass. Label errors become <i>missing</i> or <i>shifted</i> boxes (section 8).</li>
<li><b>Segmentation:</b> the same for masks, plus a new error class: masks that are the wrong shape
(swollen, shifted, crude) while the label is right (section 9).</li>
</ul>
Deduplicate once, before splitting, for all three; the label checks differ per task.
</font></td></tr></table>

**Takeaway.** Four cheap checks before any training: MD5 for exact copies, a perceptual hash with a
threshold you have looked at for near copies, the same hash on flipped and rotated variants, and a
"does it open, what mode, what size" pass for broken files. Then sort each class by how unusual its photos
are and let a human read the top of the list. The output is a table, `clean_images`, and one line you can
put in a report: removed N, kept M. Section 6 splits that table without leaking.

## 6. Splits & leakage

**Objective:** choose a split that does not leak, and understand what k-fold, stratified, leave-one-out
and leave-one-group-out cross-validation each buy you.

Section 5 ended with `clean_images`, a table without duplicates. The next decision is the one that decides
whether next week's test score means anything: **which photos does the model never see?** The held-out
part has to look like the photos the park's cameras will take *after* training, not like the ones they
already took. If a test photo has a near-twin in training, the score reports memory, not recognition.

<table width="100%"><tr><td bgcolor="#E3F2FD"><font color="#0D47A1">
<b>📘 Theory · Data leakage</b><br>
<b>Leakage</b> is any way information about the test photos reaches the model during training. The
obvious route is a duplicate in both halves (section 5). The quiet route is <b>a photo that is not a copy
but comes from the same moment</b>: two frames of one burst, two shots of the same animal in front of the
same bush, two photos from one camera that always sees the same tree. The model learns the bush, the
score goes up, and the first camera in a new place breaks it. The rule: <b>split by whatever will be new
at deployment</b> (a camera, a day, a patient, a farm), not by photo.
</font></td></tr></table>

### Random vs stratified split on the Play Dataset

The simplest split shuffles the photos and cuts them 80 / 20. A **stratified** split cuts *each class*
80 / 20, so both halves keep the class shares of the whole table. The code below does both on
`clean_images` and compares the class shares of the test halves.

<table width="100%"><tr><td bgcolor="#E0F7F4"><font color="#00564D">
<b>❓ Questions</b><br>
Sheep is the smallest class, with a fifth of the photos of cat. Think of a random 20 % test half.
</font></td></tr></table>

**Q15.** 🟡 In a random split, which class share will move the most between train and test: the largest
class or the smallest? Does the stratified split fix that completely, or only mostly? Decide, then run.

In [ ]:
# Random split vs stratified split of clean_images (80 / 20), then the class share of each test half.
from sklearn.model_selection import (
    GroupKFold,
    GroupShuffleSplit,
    KFold,
    LeaveOneGroupOut,
    LeaveOneOut,
    StratifiedKFold,
    train_test_split,
)

TEST_SIZE = 0.2
_ids = clean_images["id"].to_numpy()
_y = clean_images["class"].to_numpy()
_tr_rand, _te_rand = train_test_split(_ids, test_size=TEST_SIZE, random_state=SEED)
_tr_strat, _te_strat = train_test_split(_ids, test_size=TEST_SIZE, random_state=SEED, stratify=_y)


def class_shares(ids) -> pd.Series:
    """Share of each class among the given image ids of clean_images (in CLASSES order, sums to 1)."""
    counts = clean_images.set_index("id").loc[list(ids), "class"].value_counts()
    return counts.reindex(CLASSES).fillna(0) / max(len(ids), 1)


shares = pd.DataFrame(
    {
        "whole table": class_shares(_ids),
        "random · test": class_shares(_te_rand),
        "stratified · test": class_shares(_te_strat),
    }
)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, col in zip(axes, ["random · test", "stratified · test"], strict=True):
    x = np.arange(len(CLASSES))
    ax.bar(x - 0.2, shares["whole table"], width=0.4, color="#DDE5E7", label="whole table")
    ax.bar(x + 0.2, shares[col], width=0.4, color=[CLASS_COLORS[c] for c in CLASSES], label=col)
    ax.set_xticks(x, CLASSES)
    ax.set_title(f"{col} ({len(_te_rand)} photos)")
    ax.set_ylabel("share of photos")
    ax.legend(frameon=False)
plt.tight_layout()
plt.show()
_drift = (shares.sub(shares["whole table"], axis=0).abs() * 100).round(1)
print("largest class-share drift from the whole table, in percentage points:")
print(f"  random split: {_drift['random · test'].max():.1f} ({_drift['random · test'].idxmax()})   "
      f"stratified split: {_drift['stratified · test'].max():.1f} ({_drift['stratified · test'].idxmax()})")
RESULTS["split_drift_random_pp"] = float(_drift["random · test"].max())
RESULTS["split_drift_stratified_pp"] = float(_drift["stratified · test"].max())

💡 The grey bars are the whole table and the coloured bars the test half: with a random cut the small
classes wobble the most in relative terms, and the stratified cut sits on top of the grey bars up to the
rounding that whole photos force.

**Worked answer (Q15).** In absolute percentage points the largest classes can move the most, but what
hurts is the *relative* move: a class of 50 photos that loses or gains five in the test half changes its
share by 10 %, and with a rare class like the paddock's dog the test half may end up with almost none of
the one class the alert is about. Stratification removes that wobble up to rounding (you cannot put 10.6
sheep in the test half), so the shares match to within one photo per class. It does nothing about the
content of the photos, which is the problem the rest of this section is about.

### Cross-validation: four ways to reuse one dataset

One split gives one score, and with a few hundred photos that score moves by several points when you
change the seed. **Cross-validation** rotates the test half around the table so every photo is tested
once, and reports the mean (and spread) over the folds. Four schemes, each with its reason:

<table width="100%"><tr><td bgcolor="#E3F2FD"><font color="#0D47A1">
<b>📘 Theory · k-fold, stratified k-fold, leave-one-out, leave-one-group-out</b><br>
<ul>
<li><b>k-fold:</b> cut the table into k pieces, test on each piece once, train on the other k−1.
Five or ten folds is the usual choice.</li>
<li><b>stratified k-fold:</b> the same, but every piece keeps the class shares. Use it whenever classes are
imbalanced, which is always.</li>
<li><b>leave-one-out (LOO):</b> k equals the number of photos. Every photo is tested once against a model
trained on all the others. Useful when the dataset is <b>very small</b> (a few dozen photos) and every
training example counts; otherwise it costs n trainings, and the n scores are each 0 or 1, so the mean is
noisy.</li>
<li><b>leave-one-group-out (LOGO):</b> the folds are <b>groups</b> that must stay together (one camera, one
day, one patient). Each fold holds out one whole group, which is the only scheme here that tests what
happens at a <i>new</i> camera.</li>
</ul>
</font></td></tr></table>

The cell below runs the first three on the Play Dataset and reports, for each, how many folds there are,
how big a test fold is, and the smallest class share any test fold has (0 means some fold tests a class
it never sees). LOO runs on a small slice of the table, because that is where it belongs.

**Q16.** 🟡 Before running: how many folds does leave-one-out make on the Play Dataset, and what is the
smallest class share a LOO test fold can have?

In [ ]:
# k-fold, stratified k-fold and leave-one-out on the Play Dataset: folds, test size, smallest class share.
K_FOLDS = 5
LOO_N = 12 if SMOKE else 40  # leave-one-out on a slice: a "very small dataset" (N-014: same code, fewer rows)


def summarise_folds(name: str, splitter, X, y, groups=None) -> dict:
    """One row per CV scheme: folds, test-fold size and the smallest class share across the test folds."""
    sizes, min_share = [], 1.0
    for _, test_idx in splitter.split(X, y, groups):
        sizes.append(len(test_idx))
        counts = pd.Series(y[test_idx]).value_counts(normalize=True).reindex(sorted(set(y))).fillna(0)
        min_share = min(min_share, float(counts.min()))
    return {"scheme": name, "folds": len(sizes), "test fold size": f"{min(sizes)}–{max(sizes)}",
            "smallest class share in a test fold": round(min_share, 3)}


_loo_ids = _ids[:LOO_N]
cv_rows = [
    summarise_folds("kfold", KFold(K_FOLDS, shuffle=True, random_state=SEED), _ids, _y),
    summarise_folds("stratified_kfold", StratifiedKFold(K_FOLDS, shuffle=True, random_state=SEED), _ids, _y),
    summarise_folds("loo", LeaveOneOut(), _loo_ids, _y[:LOO_N]),
]
cv_schemes = [r["scheme"] for r in cv_rows]
display(pd.DataFrame(cv_rows).set_index("scheme"))
print(f"(loo ran on the first {LOO_N} photos: {LOO_N} folds, {LOO_N} trainings for one score)")

💡 Plain k-fold already drops the smallest class's share in some fold below its overall share; stratified
k-fold holds it steady; LOO's test folds are one photo each, so the share column reads 0 whatever you do.

**Worked answer (Q16).** LOO makes one fold per photo, so as many folds as rows, and it would cost that many
trainings next week. Each test fold is a single photo, so its smallest class share is always 0: five of
six classes are absent from every fold. That is harmless for the mean score but shows why LOO tells you
nothing per class. The fourth scheme, leave-one-group-out, needs groups, and the Play Dataset has none:
COCO photos come from thousands of unrelated Flickr accounts. The park's next cameras do have groups.

### The woodland trail cameras

The park has just hung **trail cameras in its woodland reserve**. A trail camera fires on motion and takes
a **burst** of three frames a second apart, every night, from the same tree, for years. We use a slice of
the Caltech Camera Traps dataset (Beery et al., 2018) as a stand-in: eight camera locations, five
animals (opossum, raccoon, coyote, bobcat, rabbit), whole bursts kept together. `data/cct20_slice.csv`
lists the photos; the next cell downloads them from LILA's public bucket and shrinks them to 512 px, so
**the first run takes a minute** and afterwards the cell is instant. The five woodland classes borrow the
park's six colours so every plot keeps one colour per class.

In [ ]:
# @title Trail-camera helpers (manifest, download + resize to 512 px, burst gallery) { display-mode: "form" }
CCT_DIR = DATA_DIR / "cache" / "cct20"
CCT_SIDE = 512
CCT_CLASSES = ["opossum", "raccoon", "coyote", "bobcat", "rabbit"]
CCT_COLORS = dict(zip(CCT_CLASSES, [CLASS_COLORS[c] for c in CLASSES], strict=False))
SMOKE_CCT_LOCATIONS = 3
SMOKE_CCT_IMAGES = 30


def select_cct_rows(table: pd.DataFrame) -> pd.DataFrame:
    """The slice to use: everything, or in smoke mode whole bursts from 3 locations up to 30 photos."""
    if not SMOKE:
        return table
    keep = []
    locations = sorted(table["location"].unique())[:SMOKE_CCT_LOCATIONS]
    per_location = SMOKE_CCT_IMAGES // SMOKE_CCT_LOCATIONS
    for loc in locations:
        n = 0
        at_loc = table[table["location"] == loc].sort_values(["seq_id", "frame_num"])
        for _, burst in at_loc.groupby("seq_id", sort=True):
            if n + len(burst) > per_location:
                continue
            keep.append(burst)
            n += len(burst)
    return pd.concat(keep).reset_index(drop=True)


def fetch_cct_image(row: dict) -> tuple[str, Path | None]:
    """Download one trail-camera photo and store a 512 px copy; returns (file_name, path or None)."""
    dest = CCT_DIR / row["file_name"]
    if dest.exists() and dest.stat().st_size > 0:
        return row["file_name"], dest
    raw = download_file(row["url"], CCT_DIR / "raw" / row["file_name"])
    if raw is None:
        return row["file_name"], None
    with Image.open(raw) as im:
        im.draft("RGB", (CCT_SIDE, CCT_SIDE))
        small = im.convert("RGB")
        small.thumbnail((CCT_SIDE, CCT_SIDE))
        small.save(dest, "JPEG", quality=90)
    raw.unlink(missing_ok=True)
    return row["file_name"], dest


def download_cct(table: pd.DataFrame, workers: int = 16) -> pd.DataFrame:
    """Fetch every photo of `table` (parallel, cached), return the rows whose photo is on disk with a path."""
    CCT_DIR.mkdir(parents=True, exist_ok=True)
    paths, done = {}, 0
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for name, path in pool.map(fetch_cct_image, table.to_dict("records")):
            done += 1
            if path is not None:
                paths[name] = str(path)
            print(f"\r  trail-camera photos ready: {done}/{len(table)}", end="", flush=True)
    print()
    out = table[table["file_name"].isin(paths)].copy()
    out["path"] = out["file_name"].map(paths)
    return out.sort_values(["location", "seq_id", "frame_num"]).reset_index(drop=True)


def show_bursts(table: pd.DataFrame, n_bursts: int, size: int = 256) -> None:
    """One row per burst: its frames side by side, framed in the class colour, titled with the location."""
    bursts = [b for _, b in table.groupby("seq_id", sort=True) if len(b) > 1][:n_bursts]
    cols = max(len(b) for b in bursts)
    fig, axes = plt.subplots(len(bursts), cols, figsize=(2.6 * cols, 2.1 * len(bursts)), squeeze=False)
    for r, burst in enumerate(bursts):
        for c in range(cols):
            ax = axes[r, c]
            if c < len(burst):
                row = burst.iloc[c]
                ax.imshow(load_thumb(row["path"], size))
                frame(ax, CCT_COLORS[row["label"]])
                ax.set_title(f"camera {row['location']} · {row['label']} · frame {row['frame_num']}",
                             fontsize=8)
            else:
                ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# Load the slice, download and shrink the photos (cached), then look at a few bursts.
cct_all = pd.read_csv(fetch_manifest("cct20_slice.csv"))
cct = download_cct(select_cct_rows(cct_all))
if len(cct) < len(select_cct_rows(cct_all)):
    print(f"  ⚠️ {len(select_cct_rows(cct_all)) - len(cct)} photos could not be downloaded; run again.")
CCT_LOCATIONS = sorted(cct["location"].unique().tolist())
print(f"{len(cct)} photos · {cct['seq_id'].nunique()} bursts · cameras {CCT_LOCATIONS}")
print("photos per class: " + " · ".join(f"{c} {int((cct['label'] == c).sum())}" for c in CCT_CLASSES))
show_bursts(cct, n_bursts=3)

💡 The frames of a burst differ by one second and a few steps of the animal; everything else, the tree,
the ground and the exposure, is identical. If one frame is in test and its siblings in train, the model is
asked to recognise a photo it has almost seen.

### Leakage, measured without a model

No training is needed to see the leak. For any split, count **the share of test photos that have a burst
sibling (same `seq_id`) in training**. A random split scatters the frames of each burst over both halves;
a split **by camera** keeps every burst, and every tree, on one side.

**Q17.** 🟡 Predict the sibling share of a random 80 / 20 split of the trail-camera photos: about 0 %,
about 20 %, or most of the test photos? And of a split by camera?

In [ ]:
# Share of test photos with a burst sibling in training: random split vs split by camera.
def sibling_rate(train: pd.DataFrame, test: pd.DataFrame) -> float:
    """Share of test photos whose seq_id (burst) also appears in the training table (0 = no leak)."""
    if len(test) == 0:
        return 0.0
    return float(test["seq_id"].isin(set(train["seq_id"])).mean())


_tr, _te = train_test_split(cct.index.to_numpy(), test_size=TEST_SIZE, random_state=SEED)
sibling_random = sibling_rate(cct.loc[_tr], cct.loc[_te])

_gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
_tr_g, _te_g = next(_gss.split(cct, groups=cct["location"]))
sibling_camera = sibling_rate(cct.iloc[_tr_g], cct.iloc[_te_g])
_held_out = sorted(cct.iloc[_te_g]["location"].unique().tolist())

RESULTS["sibling_rate_random"] = sibling_random
RESULTS["sibling_rate_by_camera"] = sibling_camera
print(f"random split    : {sibling_random:5.1%} of the {len(_te)} test photos have a burst sibling in train")
print(f"split by camera : {sibling_camera:5.1%} of the {len(_te_g)} test photos (held out: {_held_out})")

fig, ax = plt.subplots(figsize=(5.5, 3))
ax.bar(["random split", "split by camera"], [sibling_random, sibling_camera],
       color=[CLASS_COLORS["cat"], CLASS_COLORS["zebra"]])
ax.set_ylabel("test photos with a burst sibling in train")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
plt.tight_layout()
plt.show()

💡 With bursts of three frames, a random split gives most test photos a sibling in training; splitting by
camera brings the share to zero by construction, because a burst never spans two cameras.

**Worked answer (Q17).** A random split leaks for most test photos: a frame lands in test, and with high
probability at least one of its two siblings landed in train. The split by camera leaks for none. The same
effect shows in the paper the data comes from: Beery et al. (2018) report a classification **error of
20.8 % at cameras seen in training and 41.1 % at cameras never seen**, doubling at the first new tree. A
random split would have reported the 20.8 % and the park would have discovered the 41.1 % in the field.

<table width="100%"><tr><td bgcolor="#FDECEA"><font color="#8B1A10">
<b>⚠️ Pitfall · Leave-one-out leaks like a random split</b><br>
LOO sounds strict (train on everything but one photo), but on burst data it is the <i>worst</i> case: the
held-out frame's siblings are always in training. The fix is <b>leave-one-<i>group</i>-out</b> with the
camera as the group: every fold holds out one whole camera, so the sibling share is zero in every fold
and the score is the one a new camera will see. The same holds for any "one item at a time" scheme: the
unit you leave out must be the unit that will be new.
</font></td></tr></table>

The cell below checks that claim fold by fold: leave-one-out and leave-one-group-out on the trail-camera
photos, each reporting the mean sibling share over its folds.

In [ ]:
# Leave-one-out vs leave-one-group-out (camera) on the trail cameras: sibling share per fold.
def mean_sibling_rate(splitter, table: pd.DataFrame, groups=None) -> float:
    """Mean over the folds of the share of test photos with a burst sibling in the fold's training part."""
    rates = [sibling_rate(table.iloc[tr], table.iloc[te]) for tr, te in splitter.split(table, groups=groups)]
    return float(np.mean(rates))


loo_sibling = mean_sibling_rate(LeaveOneOut(), cct)
logo_sibling = mean_sibling_rate(LeaveOneGroupOut(), cct, groups=cct["location"])
cv_schemes += ["logo"]
RESULTS["cv_schemes"] = cv_schemes
RESULTS["sibling_rate_loo"] = loo_sibling
RESULTS["sibling_rate_logo"] = logo_sibling
print(f"leave-one-out        : {len(cct)} folds, sibling share {loo_sibling:.1%}")
print(f"leave-one-group-out  : {cct['location'].nunique()} folds (one per camera), sibling share "
      f"{logo_sibling:.1%}")
print(f"cross-validation schemes run in this notebook: {cv_schemes}")

💡 LOO's sibling share is the share of photos that belong to a burst of more than one frame, which is
nearly all of them; LOGO by camera is zero in every fold.

### ✍️ Choose and run a split

<table width="100%"><tr><td bgcolor="#FFF4D6"><font color="#7A4A00">
<b>✏️ Your turn · <code>split_by_camera</code></b><br>
Fill in the function: given the trail-camera table and a list of camera locations to hold out, return a
dictionary <code>{"train": [...], "test": [...]}</code> of <code>file_name</code> values, every photo of a
held-out camera in test and every other photo in train, each list sorted. Then choose
<code>TEST_LOCATIONS</code> (any cameras from <code>CCT_LOCATIONS</code>). The <code>use()</code> line
checks your function against the reference and continues with the reference either way.
</font></td></tr></table>

In [ ]:
# ✍️ YOUR CODE
def split_by_camera(table, test_locations):
    """Split the trail-camera table by camera.

    table: a DataFrame with columns file_name and location (cct). test_locations: list of location ids.
    Returns {"train": sorted file_names of every other camera, "test": sorted file_names of test_locations}.
    """
    ...


def _ref_split_by_camera(table, test_locations):
    in_test = table["location"].isin(test_locations)
    return {"train": sorted(table.loc[~in_test, "file_name"]),
            "test": sorted(table.loc[in_test, "file_name"])}


TEST_LOCATIONS = [CCT_LOCATIONS[0], CCT_LOCATIONS[-1]]  # choose: any cameras from CCT_LOCATIONS
_bad = [loc for loc in TEST_LOCATIONS if loc not in CCT_LOCATIONS]
if _bad or not TEST_LOCATIONS or len(set(TEST_LOCATIONS)) >= len(CCT_LOCATIONS):
    raise SystemExit(f"TEST_LOCATIONS must be a non-empty subset of {CCT_LOCATIONS} that leaves cameras for "
                     f"training; unknown: {_bad}")

my_split = use(split_by_camera, cct, TEST_LOCATIONS, reference=_ref_split_by_camera)
_train_t = cct[cct["file_name"].isin(my_split["train"])]
_test_t = cct[cct["file_name"].isin(my_split["test"])]
print(f"train: {len(_train_t)} photos from cameras {sorted(_train_t['location'].unique().tolist())}")
print(f"test : {len(_test_t)} photos from cameras {sorted(_test_t['location'].unique().tolist())}")
print(f"sibling share of this split: {sibling_rate(_train_t, _test_t):.1%}")
_split_shares = pd.DataFrame({
    "train": _train_t["label"].value_counts(normalize=True).reindex(CCT_CLASSES).fillna(0),
    "test": _test_t["label"].value_counts(normalize=True).reindex(CCT_CLASSES).fillna(0),
}).round(2)
display(_split_shares.T)
_absent = [c for c in CCT_CLASSES if _split_shares.loc[c, "test"] == 0]
if _absent:
    print(f"⚠️ no {', '.join(_absent)} photos in the test cameras: those classes get no score. "
          "Try other cameras.")

💡 A split by camera has no siblings across the cut, but it can leave a class without test photos
(not every animal walks past every camera): you trade a clean score for a less balanced one, and you choose
the held-out cameras with both in mind.

### 🔴 Optional: a ruler to see the gap

<table width="100%"><tr><td bgcolor="#E3F2FD"><font color="#0D47A1">
<b>📘 Theory · The nearest-neighbour ruler (not training)</b><br>
No network is trained in this notebook. To <i>see</i> the gap the two splits create, we use a
<b>ruler</b>: a 1-nearest-neighbour classifier on each photo's colour histogram (the 96 numbers section 3
computed). It has no parameters to learn, runs in seconds on a CPU, and is nowhere near a real model. Its
only job is to measure the same thing twice: once with k-fold (random, leaks across bursts) and once with
leave-one-camera-out.
</font></td></tr></table>

**Q18.** 🔴 Will the ruler score higher with k-fold or with leave-one-camera-out, and roughly by how much?
Predict, then run.

In [ ]:
# The colour-histogram ruler: 1-NN accuracy under k-fold vs leave-one-camera-out on the trail cameras.
from sklearn.neighbors import KNeighborsClassifier

RULER_THUMB = 64
_features = np.stack([measure_image(p, RULER_THUMB)[1].ravel() for p in cct["path"]])
_labels = cct["label"].to_numpy()


def ruler_accuracy(splitter, groups=None) -> float:
    """Mean 1-NN accuracy over the folds of `splitter` on the colour-histogram features."""
    scores = []
    for tr, te in splitter.split(_features, _labels, groups):
        knn = KNeighborsClassifier(n_neighbors=1).fit(_features[tr], _labels[tr])
        scores.append(float((knn.predict(_features[te]) == _labels[te]).mean()))
    return float(np.mean(scores))


_n_cameras = cct["location"].nunique()
ruler_kfold = ruler_accuracy(KFold(min(K_FOLDS, _n_cameras), shuffle=True, random_state=SEED))
ruler_by_camera = ruler_accuracy(GroupKFold(_n_cameras), groups=cct["location"])
RESULTS["ruler_kfold"] = ruler_kfold
RESULTS["ruler_by_camera"] = ruler_by_camera
print(f"ruler accuracy, k-fold (random)      : {ruler_kfold:.1%}")
print(f"ruler accuracy, leave-one-camera-out : {ruler_by_camera:.1%}")
print(f"chance level with {len(CCT_CLASSES)} classes: {1 / len(CCT_CLASSES):.0%}")

fig, ax = plt.subplots(figsize=(5.5, 3))
ax.bar(["k-fold (random)", "leave-one-camera-out"], [ruler_kfold, ruler_by_camera],
       color=[CLASS_COLORS["cat"], CLASS_COLORS["zebra"]])
ax.axhline(1 / len(CCT_CLASSES), color=INK, linestyle="--", linewidth=1, label="chance")
ax.set_ylabel("1-NN ruler accuracy")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

💡 The ruler scores by finding the most similar photo in training: under k-fold that photo is usually a
burst sibling with the same animal in front of the same tree, so it looks good; under leave-one-camera-out
it has to find a similar photo from another camera and falls towards chance.

**Worked answer (Q18).** The random scheme scores higher, and the gap is large: on the full slice the ruler
roughly doubles its accuracy when its neighbours are allowed to be siblings. The exact numbers depend on
the slice and the seed, so read the direction and the size of the gap, not the digits. A real model would
score higher on both, and the gap would still be there: that is the 20.8 % → 41.1 % of the paper.

<table width="100%"><tr><td bgcolor="#EDE7F6"><font color="#311B92">
<b>🔀 Across tasks · Splitting for boxes and masks</b><br>
<ul>
<li><b>Classification:</b> stratify by the image label; group by camera, day or source.</li>
<li><b>Detection:</b> the unit is still the <i>image</i> (never split boxes of one photo across halves),
but one photo carries several labels, so plain stratification does not apply. <i>Iterative
stratification</i> balances the box counts per class across folds (section 8).</li>
<li><b>Segmentation:</b> the same, with pixel shares instead of box counts; a rare class with a few
large masks can end up entirely in one fold if you do not check (section 9).</li>
</ul>
In all three, deduplicate first (section 5), then split, then never look at the test half again.
</font></td></tr></table>

**Takeaway.** A split decides what the score means. Stratify so every class is scored; group by whatever
will be new when the model is deployed, so the score is the one you will see there; and measure leakage
before any model with one number, the share of test photos with a sibling in training. Leave-one-out is
for tiny datasets and is no protection against siblings; leave-one-group-out is. Section 7 picks the
metric to compute on that test half.